# 19 · CP or Tucker in practice / CP o Tucker en la práctica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/19-cp-or-tucker-in-practice.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2">DEEP DIVE · TAKE-HOME · ABOUT 2 HOURS / ESTUDIO A FONDO · PARA DESPUÉS · UNAS 2 HORAS</span>

Sections 10 and 11 fitted Tucker and CP to a small taxi tensor and compared how many numbers each stored. This deep dive takes the same two models to three tensors people factorise for a living, and asks which one to choose:

> **CP and Tucker differ in one object, the core tensor, and the right core depends on whether the factorisation has to predict entries nobody observed or store a known tensor in fewer numbers.**

On the FB15k-237 knowledge graph the job is prediction, and the question is what a core can *express*. On a trained ResNet-18 convolution kernel the job is storage, and the question is what a core *costs* — and what CP's savings hide. Then the question both leave open, which rank to fit. Last, a photograph that can be asked either job, and ranks the two models in opposite orders.

It works through the ML blog post *CP or Tucker in practice* at a first-read pace, with the heaviest experiments cut to run in a few minutes on Colab's CPU.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">Las secciones 10 y 11 ajustaron Tucker y CP a un tensor pequeño de taxis y compararon cuántos números guardaba cada uno. Este estudio a fondo lleva los mismos dos modelos a tres tensores que se factorizan de verdad y pregunta cuál elegir:</div><div style="margin:0 0 .7em"><b>CP y Tucker difieren en un solo objeto, el tensor núcleo, y el núcleo correcto depende de si la factorización tiene que predecir entradas que nadie observó o guardar un tensor conocido con menos números.</b></div><div style="margin:0 0 .7em">En el grafo de conocimiento FB15k-237 el trabajo es predecir, y la pregunta es qué puede <b>expresar</b> un núcleo. En un kernel de convolución entrenado de ResNet-18 el trabajo es guardar, y la pregunta es lo que <b>cuesta</b> un núcleo y lo que esconde el ahorro de CP. Después, la pregunta que ambos dejan abierta: qué rango ajustar. Por último, una fotografía a la que se le pueden pedir los dos trabajos, y que ordena los dos modelos al revés.</div><div style="margin:0 0 0">Sigue la entrada del blog «CP or Tucker in practice» (en inglés) a ritmo de primera lectura, con los experimentos más pesados recortados para la CPU de Colab.</div></div>

## What you will be able to do / Lo que podrás hacer

- Write CP as **Tucker with a superdiagonal core**, and count what each stores.
- Explain why **Tucker's factors** are defined only up to a change of basis and **CP's** are not.
- Measure from real triples how often a knowledge-graph relation is **reversed**, and how often a symmetric score gets the chance to be wrong.
- Write **DistMult, RESCAL and TuckER** as one scoring function with three cores, and show DistMult cannot learn a direction.
- Implement a CP-compressed convolution as **four layers** and Tucker-2 as three, and verify the wiring against the dense layer.
- Compare CP and Tucker-2 at **matched parameter budgets**, and read CP's advantage as **cancellation** between large terms.
- Read ranks off an **HOSVD scree**, check its truncation bound, and say when a spectrum cannot choose a rank.
- Find a CP rank by **over-parameterising and pruning** with a group-lasso proximal step.
- Fold a photograph into a 4-way tensor and show that the model that **compresses** best is not the one that **predicts** best.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Escribir CP como <b>Tucker con núcleo superdiagonal</b> y contar lo que guarda cada uno.</li><li style="margin:.35em 0">Explicar por qué los <b>factores de Tucker</b> solo están definidos salvo un cambio de base y los de <b>CP</b> no.</li><li style="margin:.35em 0">Medir con tripletas reales con qué frecuencia una relación de un grafo de conocimiento está <b>invertida</b>, y cuántas veces una puntuación simétrica tiene ocasión de equivocarse.</li><li style="margin:.35em 0">Escribir <b>DistMult, RESCAL y TuckER</b> como una sola función de puntuación con tres núcleos, y mostrar que DistMult no puede aprender un sentido.</li><li style="margin:.35em 0">Programar una convolución comprimida con CP como <b>cuatro capas</b> y Tucker-2 como tres, y comprobar el cableado frente a la capa densa.</li><li style="margin:.35em 0">Comparar CP y Tucker-2 con <b>presupuestos de parámetros iguales</b> e interpretar la ventaja de CP como <b>cancelación</b> entre términos grandes.</li><li style="margin:.35em 0">Leer rangos en el <b>codo de HOSVD</b>, comprobar su cota de truncamiento y decir cuándo un espectro no puede elegir un rango.</li><li style="margin:.35em 0">Encontrar un rango CP <b>sobreparametrizando y podando</b> con un paso proximal de group lasso.</li><li style="margin:.35em 0">Plegar una fotografía en un tensor de 4 modos y mostrar que el modelo que mejor <b>comprime</b> no es el que mejor <b>predice</b>.</li></ul></div>

<!-- CORE-PATH -->
## Core path / Ruta esencial

Take-home, about two hours at a graduate pace. Make the **Predict first** choice below, run **Setup**, then read Part 1 and do **Exercise 1**, the Core activity. Parts 2–5 each end in their own exercise with a folded solution; do them in order, because each part reuses what the one before it built.

**You can:**

- Say what the one structural difference between CP and Tucker is, and count what it costs.
- Predict, before fitting anything, which of the two a given job — scoring unseen facts, storing a known tensor, filling in hidden entries — is likely to favour, and say what you would measure to check.

<details>
<summary>Español · Ruta y metas</summary>

Para después del taller, unas dos horas a ritmo de posgrado. Responde primero a **Predice primero**, ejecuta **Preparación**, lee la Parte 1 y haz el **Ejercicio 1**, la actividad esencial. Las Partes 2–5 terminan cada una con su ejercicio y una solución plegada; hazlas en orden, porque cada parte reutiliza lo que construyó la anterior.

**Al terminar puedes:**

- Decir cuál es la única diferencia estructural entre CP y Tucker, y contar lo que cuesta.
- Predecir, antes de ajustar nada, qué trabajo — puntuar hechos no vistos, guardar un tensor conocido, rellenar entradas ocultas — favorece a cada uno, y decir qué medirías para comprobarlo.

</details>

## Predict first / Predice primero

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

A knowledge graph stores facts as triples: *(subject, relation, object)*. DistMult, one of the most-cited embedding models, scores a triple with one vector per entity and one per relation. Someone says the following. **Decide whether they are right before you reveal anything.**

<div style="border-left:5px solid #0891b2;background:rgba(8,145,178,.10);border-radius:0 8px 8px 0;padding:18px 22px;margin:2.2em 0;font:400 15px/1.85 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#0891b2;margin-bottom:11px">THE CLAIM · LA AFIRMACIÓN</div>&ldquo;DistMult scores <code>(Paris, contains, France)</code> as high as <code>(France, contains, Paris)</code> at the start of training, but that is just a bad initialisation. <b>Train it long enough</b> on enough facts and it will learn which way round containment goes.&rdquo;</div>

A prediction you have committed to is worth more than one you keep adjusting as the answer appears.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un grafo de conocimiento guarda hechos como tripletas: <i>(sujeto, relación, objeto)</i>. DistMult puntúa una tripleta con un vector por entidad y uno por relación. Alguien afirma que DistMult puntúa igual <code>(París, contiene, Francia)</code> y <code>(Francia, contiene, París)</code> solo al principio, y que <b>entrenando lo suficiente</b> aprenderá en qué sentido va la contención. Decide si tiene razón <b>antes</b> de revelar la comprobación.</div>

In [ ]:
#@title 🤔 Predict: can training teach DistMult which way "contains" goes? / Predice: ¿puede el entrenamiento enseñar a DistMult el sentido de "contiene"? — run me / ejecútame { display-mode: 'form' }

# --- counterexample / contraejemplo
import numpy as np
rng_claim = np.random.default_rng(19)
e_paris, e_france, w_contains = rng_claim.standard_normal((3, 8))
france_contains_paris = np.sum(e_france * w_contains * e_paris)
paris_contains_france = np.sum(e_paris * w_contains * e_france)
assert np.isclose(france_contains_paris, paris_contains_france)
# --- end counterexample

import ipywidgets as widgets
from IPython.display import display

# --- how the question is laid out / cómo se presenta la pregunta ---
# Radio buttons rather than a dropdown: one bilingual answer per line, with
# room around them, because the whole point is weighing the options against
# each other. Botones de opción en vez de un desplegable.
import contextlib
import html as pred_html
import io

PRED_ACCENT = "#0891b2"
PRED_SANS = "ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"
PRED_MONO = "ui-monospace,SFMono-Regular,Menlo,Consolas,monospace"


def pred_tag(text):
    """A small EN / ES marker, in words rather than in colour alone."""
    return (f'<span style="font:700 10px/1 {PRED_MONO};letter-spacing:.16em;'
            f'color:{PRED_ACCENT};opacity:.8;margin-right:9px;'
            f'vertical-align:.12em">{text}</span>')


def pred_is_measurement(line):
    """True for a printed reading, false for a sentence."""
    if ":" not in line:
        return False
    tail = line.rsplit(":", 1)[1].strip()
    return bool(tail) and (tail[0].isdigit()
                           or tail[0] in "[(-+."
                           or tail.startswith(("True", "False", "nan", "inf")))


def pred_panel(text):
    """The reveal, laid out instead of printed. EN and ES stay written tags."""
    blocks = []
    for line in text.rstrip("\n").split("\n"):
        stripped = line.strip()
        if not stripped:
            blocks.append('<div style="height:12px"></div>')
        elif stripped.startswith(("EN:", "ES:")):
            tag, body = stripped[:2], stripped[3:].strip()
            blocks.append(
                f'<p style="margin:.55em 0;font:400 15px/1.8 {PRED_SANS}">'
                f'{pred_tag(tag)}{pred_html.escape(body)}</p>')
        elif pred_is_measurement(stripped):
            blocks.append(
                f'<div style="font:400 13.5px/2.0 {PRED_MONO};'
                f'white-space:pre-wrap">{pred_html.escape(stripped)}</div>')
        else:
            blocks.append(
                f'<p style="margin:.55em 0;font:600 15.5px/1.75 {PRED_SANS}">'
                f'{pred_html.escape(stripped)}</p>')
    return (f'<div style="border-left:4px solid {PRED_ACCENT};'
            f'background:rgba(130,130,150,.08);border-radius:0 10px 10px 0;'
            f'padding:16px 20px;margin:.4em 0 0">{"".join(blocks)}</div>')


def pred_render(choice, reveal):
    """Run the check, catch what it prints, and show it laid out."""
    caught = io.StringIO()
    with contextlib.redirect_stdout(caught):
        check_prediction(choice, reveal)
    display(widgets.HTML(pred_panel(caught.getvalue())))


pred_choice = widgets.RadioButtons(
    options=[
        ("— choose one / elige una —", None),
        ("Yes — with enough facts, training breaks the tie / Sí — con suficientes hechos, el entrenamiento rompe el empate", 'training'),
        ("No — the two scores are equal for every possible embedding / No — las dos puntuaciones son iguales para cualquier embedding", 'never'),
        ("Only with a larger embedding dimension / Solo con una dimensión de embedding mayor", 'dimension'),
    ],
    value=None,
    description="",
    layout=widgets.Layout(width="auto", margin="0 0 6px 0"),
)

pred_reveal = widgets.Checkbox(
    value=False,
    description="Show me the answer / Muéstrame la respuesta",
    indent=False,
    layout=widgets.Layout(margin="10px 0 4px 0"),
)


def check_prediction(choice, reveal):
    if choice is None:
        print("Choose an answer first / Elige una respuesta primero.")
        return

    if not reveal:
        print("Answer saved / Respuesta guardada.")
        print("Tick the box above when you are ready / Marca la casilla de arriba cuando quieras.")
        return

    print("Random 8-dimensional embeddings / Embeddings aleatorios de dimensión 8")
    print("score(France, contains, Paris):", round(float(france_contains_paris), 6))
    print("score(Paris, contains, France):", round(float(paris_contains_france), 6))
    print()
    if choice == "never":
        print("You were right / Acertaste.")
    else:
        print("You were wrong — read on / Te equivocaste; sigue leyendo.")
    print()
    print("EN: DistMult's score is a sum of products e_s[k] · w_r[k] · e_o[k]. Multiplication commutes, so swapping subject and object changes nothing — for these vectors, for any vectors, at any dimension, after any amount of training. No data can teach a model a distinction its score function cannot express.")
    print("ES: la puntuación de DistMult es una suma de productos e_s[k] · w_r[k] · e_o[k]. La multiplicación conmuta, así que intercambiar sujeto y objeto no cambia nada: para estos vectores, para cualquiera, en cualquier dimensión y tras cualquier entrenamiento. Ningún dato enseña a un modelo una distinción que su función de puntuación no puede expresar.")


# The one <style> block in these notebooks, and the markdown rule does not
# cover it: this is widget *output*, which Colab renders. Scoped to one added
# class so it can reach nothing else; without it the options still work.
pred_choice.add_class("pred-radio")

display(widgets.HTML(
    "<style>"
    ".pred-radio .widget-radio-box label{display:flex;align-items:flex-start;"
    "margin:0 0 13px;font:400 15px/1.6 " + PRED_SANS + "}"
    ".pred-radio input[type=radio]{flex:none;margin:4px 11px 0 0;"
    "transform:scale(1.15)}"
    "</style>"
))

pred_output = widgets.interactive_output(
    pred_render,
    {"choice": pred_choice, "reveal": pred_reveal},
)

pred_heading = widgets.HTML(
    f'<div style="font:700 11px/1.6 {PRED_MONO};letter-spacing:.18em;'
    f'color:{PRED_ACCENT};margin:2px 0 12px">'
    f'YOUR PREDICTION · TU PREDICCIÓN</div>'
)

display(widgets.VBox(
    [pred_heading, pred_choice, pred_reveal, pred_output],
    layout=widgets.Layout(padding="2px 0 14px 0"),
))

## What this notebook is about / De qué trata este cuaderno

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Sections 10 and 11 fitted Tucker and CP to one small taxi tensor and compared how many numbers each stored. This notebook takes the same two models to three tensors that people actually factorise for a living, and asks a sharper question: **which one should you choose, and how would you know?**

The answer this notebook argues for fits in one sentence:

> **CP and Tucker differ in one object, the core tensor, and the right core depends on whether the factorisation has to *predict* entries nobody observed or *store* a known tensor in fewer numbers.**

Those are different jobs, and on the same tensor they can rank the two models in opposite orders. That is the surprise worth two hours.

| Part | Tensor | The job | The question |
|---|---|---|---|
| 1 | a 5 × 4 × 3 toy | — | What exactly is different between CP and Tucker, and what does each cost to store? |
| 2 | FB15k-237, a knowledge graph: 14,505 × 237 × 14,505 | predict unseen facts | What can a core *express*? When does a symmetric score cost you? |
| 3 | one trained ResNet-18 convolution kernel: 256 × 256 × 3 × 3 | store a known tensor | At a fixed number of stored weights, which fits better — and at what hidden price? |
| 4 | the kernel again, and a synthetic tensor of known rank | choose the rank | Can a spectrum tell you the rank? Can a penalty find it for you? |
| 5 | a photograph folded into 4 axes | **both** | Does the model that compresses best also predict hidden pixels best? |

**What you should already have.** Einsum (section 06), least squares and the pseudoinverse (07), the SVD and truncation (09), Tucker/HOSVD (10) and CP/ALS (11). Deep dive 14 fits CP by hand; it helps, but it is not required. On the ML side you need binary cross-entropy, what an embedding is, and what a 2-D convolution layer does.

**Where this comes from.** This notebook works through the ML blog post [CP or Tucker in practice](https://project-delphi.github.io/ml-blog/posts/cp-or-tucker-in-practice/) at a pace meant for a first read, with the heaviest experiments cut down to run in a few minutes on Colab's CPU. Where the full run would take an hour, the notebook downloads the post's cached results and says so. Its prequel, [CP or Tucker](https://project-delphi.github.io/ml-blog/posts/cp-or-tucker/), derives uniqueness and rotational freedom in more depth.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Las secciones 10 y 11 ajustaron Tucker y CP a un tensor pequeño de taxis. Este cuaderno lleva los mismos dos modelos a tres tensores que se factorizan de verdad en la práctica y pregunta: <b>¿cuál elegir, y cómo saberlo?</b>
<br><br>
La tesis cabe en una frase: <b>CP y Tucker difieren en un solo objeto, el tensor núcleo, y el núcleo correcto depende de si la factorización tiene que <i>predecir</i> entradas que nadie observó o <i>guardar</i> un tensor conocido con menos números.</b> Son trabajos distintos, y sobre el mismo tensor pueden ordenar los dos modelos al revés.
<br><br>
Parte 1: la diferencia exacta y su coste. Parte 2: un grafo de conocimiento real (FB15k-237), donde el trabajo es predecir hechos. Parte 3: un kernel de convolución entrenado de ResNet-18, donde el trabajo es comprimir. Parte 4: cómo elegir el rango. Parte 5: una fotografía a la que se le piden <b>los dos</b> trabajos.
<br><br>
Necesitas einsum (06), mínimos cuadrados (07), la SVD (09), Tucker (10) y CP (11), además de entropía cruzada binaria, embeddings y capas de convolución. El cuaderno sigue la entrada del blog <i>CP or Tucker in practice</i> (en inglés), con los experimentos más pesados recortados para la CPU de Colab; cuando la ejecución completa tardaría una hora, se descargan los resultados guardados de la entrada y se dice.</div>

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Two cells. The first installs [TensorLy](https://tensorly.org/), which Colab does not ship; the second imports everything and defines the one download helper every later cell uses. Every file this notebook downloads is checked against a SHA-256 hash before it is used, so if a source ever changes under you, you get an error that says so rather than silently different numbers.

**Downloads:** the FB15k-237 training split (21 MB), one ResNet-18 kernel (2.3 MB) and four small CSVs of cached results. Everything else is computed here.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Dos celdas. La primera instala <a href="https://tensorly.org/">TensorLy</a>, que Colab no trae; la segunda importa todo y define la función de descarga que usan las demás. Cada archivo descargado se comprueba contra su hash SHA-256 antes de usarlo: si una fuente cambia, verás un error que lo dice, no números distintos en silencio.
<br><br>
<b>Descargas:</b> el conjunto de entrenamiento de FB15k-237 (21 MB), un kernel de ResNet-18 (2,3 MB) y cuatro CSV pequeños con resultados guardados.</div>

In [ ]:
%pip install -q tensorly

In [ ]:
import hashlib
import io
import time
import urllib.error
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorly as tl
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import convolve
from skimage import data as skdata
from tensorly.decomposition import parafac

tl.set_backend("numpy")
rng = np.random.default_rng(19)
torch.manual_seed(19)

ACCENT, TEAL, CORAL, MUTED = "#0891b2", "#2a9d8f", "#c45c26", "#5f6672"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 110,
})

# The blog post's cached data, pinned to one commit of its repository so the
# files cannot change under this notebook. / Datos de la entrada, fijados a un commit.
BLOG_DATA = ("https://raw.githubusercontent.com/project-delphi/ml-blog/"
             "d9f634c19b10f86efd687c8bfb78734b365705ce/"
             "posts/cp-or-tucker-in-practice/data/")
FB15K_TRAIN = ("https://raw.githubusercontent.com/ibalazevic/TuckER/"
               "23aff592115ee91e099ce490cdbea28ef026fc54/data/FB15k-237/train.txt")


def fetch_one(url, sha256, tries=3, timeout=60):
    """The bytes at `url`, checked against their SHA-256. Retries, then says why.

    Only a failure to reach the host is retried: a 404 or 403 is an answer,
    and asking again will not change it.
    """
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            break
        except urllib.error.HTTPError as error:
            if error.code not in (429, 502, 503, 504) or attempt == tries:
                raise
            time.sleep(2 ** attempt)
        except Exception as error:
            if attempt == tries:
                raise RuntimeError(
                    f"EN: could not download {url}. Check your connection and run "
                    "this cell again. / ES: no se pudo descargar el archivo. Revisa "
                    "tu conexión y vuelve a ejecutar esta celda.") from error
            time.sleep(2 ** attempt)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != sha256:
        raise ValueError(
            f"EN: {url} is not the file this notebook was written against "
            f"(sha256 {digest[:12]}…). / ES: el archivo no es el esperado.")
    return payload


def fetch(url, sha256, tries=3, timeout=60, copy=None):
    """`fetch_one`, then -- if `url` fails for any reason -- one try at `copy`,
    the workshop's own copy of the same file on its site. The SHA-256 is
    checked whichever answered, so the copy cannot pass for another file.
    """
    try:
        return fetch_one(url, sha256, tries, timeout)
    except Exception:
        if copy is None:
            raise
    payload = fetch_one(copy, sha256, 1, timeout)
    name = copy.rsplit("/", 1)[-1]
    print(f"EN: using the workshop's copy of {name}. / "
          f"ES: usando la copia del taller de {name}.")
    return payload


def rel_error(truth, estimate):
    """Relative Frobenius error, ‖truth − estimate‖ / ‖truth‖."""
    return float(np.linalg.norm(truth - estimate) / np.linalg.norm(truth))


def leading(matrix, rank):
    """The `rank` leading left singular vectors of a matrix."""
    return np.linalg.svd(matrix, full_matrices=False)[0][:, :rank]


def t32(array):
    """A NumPy array as a contiguous float32 torch tensor, for conv weights."""
    return torch.tensor(np.ascontiguousarray(array), dtype=torch.float32)


print("Imports / Importaciones: OK")

## Part 1 · One object separates them: the core / Un objeto las separa: el núcleo

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Take an $N$-way tensor $\mathcal{X} \in \mathbb{R}^{I_1 \times \cdots \times I_N}$. Both models approximate it with one **factor matrix per mode**. They differ in how the columns of those matrices are allowed to combine.

**CP at rank $R$** is a sum of $R$ rank-one terms. Term $r$ takes column $r$ of every factor and forms their outer product:

$$
\mathcal{X} \;\approx\; \sum_{r=1}^{R} a^{(1)}_r \circ a^{(2)}_r \circ \cdots \circ a^{(N)}_r .
$$

**Tucker at multilinear rank $(R_1, \dots, R_N)$** multiplies a small **core tensor** $\mathcal{G}$ by one factor per mode:

$$
\mathcal{X} \;\approx\; \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \cdots \times_N U^{(N)},
\qquad \mathcal{G} \in \mathbb{R}^{R_1 \times \cdots \times R_N} .
$$

Written entry by entry, for three modes, the two are

$$
x_{ijk} \approx \sum_{r} a_{ir}\, b_{jr}\, c_{kr}
\qquad\text{versus}\qquad
x_{ijk} \approx \sum_{p,q,s} g_{pqs}\, u_{ip}\, v_{jq}\, w_{ks} .
$$

Read the second sum as: *every* column of the first factor may meet *every* column of the second and third, and the core entry $g_{pqs}$ says how much that particular meeting contributes. In the first sum only column $r$ meets column $r$.

**So CP is Tucker with a superdiagonal core.** Set every $R_n = R$ and zero every entry of $\mathcal{G}$ except $g_{rr\cdots r}$, and the Tucker sum collapses to the CP sum. That is the whole structural difference, and everything in this notebook follows from it:

- **Storage.** CP stores $R \sum_n I_n$ numbers. Tucker stores $\prod_n R_n + \sum_n I_n R_n$, and the core's product grows fastest. A mode kept whole, $R_n = I_n$, needs no factor at all: its square orthogonal factor can be absorbed into the core, so its $I_n R_n$ term drops out.
- **Identifiability.** Tucker's factors are only defined up to an invertible change of basis per mode: replace $U^{(n)}$ by $U^{(n)} Q_n$ and the core by $\mathcal{G} \times_n Q_n^{-1}$ and the tensor does not change. CP's only freedoms are rescaling within a term and reordering the terms. That is why a CP component can be read (deep dive 14) and a Tucker factor, on its own, cannot.
- **Expressiveness.** A dense core lets the model express interactions that a superdiagonal one cannot — Part 2 is an example where that matters.

**HOSVD** is how Tucker is usually started: take the SVD of each mode-$n$ unfolding $\mathcal{X}_{(n)}$, keep its $R_n$ leading left singular vectors as $U^{(n)}$, and project to get the core, $\mathcal{G} = \mathcal{X} \times_1 U^{(1)\top} \cdots \times_N U^{(N)\top}$. HOOI then iterates from there. In this notebook HOSVD is a way to *fit* Tucker and to *read* its ranks, not a third model.

The cell below checks each of these claims on a tensor small enough to print.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ambos modelos usan una <b>matriz factor por modo</b>; difieren en cómo pueden combinarse sus columnas. En <b>CP</b> de rango R, el término r toma la columna r de cada factor y forma su producto exterior: solo la columna r se encuentra con la columna r. En <b>Tucker</b>, un <b>núcleo</b> G pequeño deja que <i>cualquier</i> columna de un factor se encuentre con <i>cualquier</i> columna de otro, y la entrada del núcleo dice cuánto aporta ese encuentro.
<br><br>
Así que <b>CP es Tucker con núcleo superdiagonal</b>. De ahí sale todo: <b>almacenamiento</b> (CP guarda R·Σ I_n números; Tucker guarda Π R_n + Σ I_n R_n, y un modo que se deja entero no necesita factor); <b>identificabilidad</b> (los factores de Tucker solo están definidos salvo un cambio de base invertible por modo, mientras que CP solo admite reescalar dentro de un término y reordenar términos: por eso una componente CP se puede leer y un factor de Tucker, solo, no); y <b>expresividad</b> (un núcleo denso expresa interacciones que uno superdiagonal no puede).
<br><br>
<b>HOSVD</b> inicia Tucker: la SVD de cada despliegue da los factores y la proyección da el núcleo. Aquí es una forma de <i>ajustar</i> Tucker y de <i>leer</i> sus rangos, no un tercer modelo.</div>

In [ ]:
# A tensor small enough to print: 5 x 4 x 3, built from R = 2 rank-one terms.
# Un tensor que cabe en pantalla: 5 x 4 x 3, construido con R = 2 términos.
R_TOY = 2
A, B, C = (rng.standard_normal((n, R_TOY)) for n in (5, 4, 3))

cp_toy = np.einsum("ir,jr,kr->ijk", A, B, C)        # sum over r of a_r ∘ b_r ∘ c_r

core = np.zeros((R_TOY, R_TOY, R_TOY))
core[np.arange(R_TOY), np.arange(R_TOY), np.arange(R_TOY)] = 1.0   # g_rrr = 1, else 0
tucker_toy = np.einsum("pqs,ip,jq,ks->ijk", core, A, B, C)

print("1. CP == Tucker with a superdiagonal core / CP == Tucker con núcleo superdiagonal:",
      np.allclose(cp_toy, tucker_toy))

# Tucker's freedom: change basis in every mode by an invertible Q_n, undo it in the core.
# La libertad de Tucker: cambia la base de cada modo con Q_n invertible y deshazlo en el núcleo.
Q1, Q2, Q3 = (rng.standard_normal((R_TOY, R_TOY)) for _ in range(3))
A_rot, B_rot, C_rot = A @ Q1, B @ Q2, C @ Q3
core_rotated = np.einsum("da,eb,fc,abc->def", *(np.linalg.inv(Q) for Q in (Q1, Q2, Q3)), core)
tucker_rotated = np.einsum("pqs,ip,jq,ks->ijk", core_rotated, A_rot, B_rot, C_rot)
print("2. Same tensor after the rotation / Mismo tensor tras la rotación:",
      np.allclose(tucker_rotated, cp_toy))
print("   Non-zero core entries before / after / Entradas no nulas del núcleo antes / después:",
      np.count_nonzero(core), "/", np.count_nonzero(np.round(core_rotated, 12)))

# CP has no such freedom: the rotated factors are not CP factors of this tensor.
# CP no tiene esa libertad: los factores rotados ya no son factores CP de este tensor.
cp_rotated = np.einsum("ir,jr,kr->ijk", A_rot, B_rot, C_rot)
print("3. CP with the rotated factors gives the same tensor / CP con los factores rotados da el mismo tensor:",
      np.allclose(cp_rotated, cp_toy))

# CP's only freedoms: rescale inside a term, or reorder the terms.
# Las únicas libertades de CP: reescalar dentro de un término o reordenarlos.
A2, B2 = A * [2.0, 1.0], B * [0.5, 1.0]
print("4. Rescaled and reordered CP is the same tensor / CP reescalado y reordenado es el mismo tensor:",
      np.allclose(np.einsum("ir,jr,kr->ijk", A2[:, ::-1], B2[:, ::-1], C[:, ::-1]), cp_toy))

![Two 3 by 3 by 3 cores fill in step by step: CP lights only the three diagonal cells g_rrr, 3 of 27; Tucker fills all 27 cells g_pqs.](https://project-delphi.github.io/tensors-workshop/images/cube-19-core.gif)

Four lines, four facts to carry into the rest of the notebook:

1. CP is a special case of Tucker. Anything CP can represent at rank $R$, Tucker can represent at ranks $(R, \dots, R)$.
2. The *same* tensor has many Tucker descriptions. The rotated core is dense — all 8 entries non-zero — yet it describes exactly the same tensor. So "core entry $g_{101}$ is large" is a statement about a basis you chose, not about the data.
3. CP does not share that freedom. Rotating the factors breaks the fit. This rigidity is what makes CP components *essentially unique* (under Kruskal's condition), and it is also, as Part 3 shows, what can make them unstable.
4. The freedoms CP does have — scale within a term, order of terms — are why libraries return normalised factors plus a vector of weights.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cuatro hechos para el resto del cuaderno: (1) CP es un caso particular de Tucker; (2) el <i>mismo</i> tensor tiene muchas descripciones de Tucker —el núcleo rotado es denso y describe exactamente el mismo tensor—, así que «la entrada g_101 es grande» habla de la base elegida, no de los datos; (3) CP no comparte esa libertad: rotar los factores rompe el ajuste, y esa rigidez es la que hace a las componentes CP <i>esencialmente únicas</i> y, como verás en la Parte 3, también la que puede volverlas inestables; (4) las libertades que sí tiene CP —escala dentro de un término y orden de los términos— explican por qué las librerías devuelven factores normalizados más un vector de pesos.</div>

In [ ]:
# Feedback helper / Comprobador: run me before the Core activity.
# Ejecútame antes de la actividad esencial.


def check_counts(cp_numbers, tucker_numbers):
    """Says what is right, what is wrong, and what to look at next."""
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    kernel_shape = (256, 256, 3, 3)
    try:
        got = cp_numbers(kernel_shape, 36)
        ok(got == 18_648,
           "CP at rank 36 on a 256×256×3×3 kernel stores 18,648 numbers: 36 × (256+256+3+3).",
           f"cp_numbers(kernel, 36) returned {got}. CP stores one column of "
           "length I_n per mode per term, so R · Σ I_n.")
        got = tucker_numbers((5, 4, 3), (2, 2, 2))
        ok(got == 32,
           "Tucker (2, 2, 2) on 5×4×3 stores 32: a 2×2×2 core (8) plus 5·2 + 4·2 + 3·2.",
           f"tucker_numbers((5,4,3), (2,2,2)) returned {got}; expected the core's "
           "8 entries plus one I_n × R_n factor per mode.")
        got = tucker_numbers(kernel_shape, (40, 43, 3, 3))
        ok(got == 36_728,
           "Tucker-2 (40, 43, 3, 3) stores 36,728: the spatial modes are kept whole, "
           "so they cost nothing beyond the core.",
           f"tucker_numbers(kernel, (40, 43, 3, 3)) returned {got}. A mode kept "
           "whole (R_n = I_n) folds its factor into the core: it stores no factor.")
        got = tucker_numbers((5, 4, 3), (5, 4, 3))
        ok(got == 60,
           "Tucker at full ranks stores exactly the dense tensor's 60 numbers.",
           f"tucker_numbers((5,4,3), (5,4,3)) returned {got}; at full rank the "
           "core *is* the tensor and no factor is stored.")
    except Exception as error:  # a learner's function can fail in any way
        notes.append(f"❌ Your function raised {type(error).__name__}: {error}")

    print("\n".join(notes))
    print()
    print("EN: the core is the term that grows as a product. Everything Part 3 "
          "measures about storage is a consequence of that one line.")
    print("ES: el núcleo es el término que crece como un producto. Todo lo que la "
          "Parte 3 mide sobre almacenamiento es consecuencia de esa línea.")

### Exercise 1 · Core activity — what each core costs / Ejercicio 1 · Actividad esencial — lo que cuesta cada núcleo

Write two functions, then use them to make a prediction you will test in Part 3.

- `cp_numbers(shape, rank)` — how many numbers CP stores at `rank` on a tensor of `shape`.
- `tucker_numbers(shape, ranks)` — the core plus the factors, where a mode kept whole ($R_n = I_n$) stores no factor.

**Predict → Run → Explain → Check**

1. **Predict.** On the 256 × 256 × 3 × 3 convolution kernel of Part 3, compare CP at rank 100 with Tucker-2 at ranks (100, 100, 3, 3) — compress the two channel modes, keep the 3 × 3 spatial modes whole. Which stores fewer numbers? Write down a guess of the ratio.
2. **Run.** Implement both functions in the cell below and compute the two counts.
3. **Explain.** In one sentence: which term makes Tucker-2 the larger one, and how does it grow with $R$?
4. **Check.** Call `check_counts(cp_numbers, tucker_numbers)`, and compare with your written prediction before opening the solution.

<details>
<summary>Español · Predice → Ejecuta → Explica → Comprueba</summary>

Escribe `cp_numbers(shape, rank)` (cuántos números guarda CP) y `tucker_numbers(shape, ranks)` (núcleo más factores; un modo entero, R_n = I_n, no guarda factor).

1. **Predice.** Sobre el kernel de 256 × 256 × 3 × 3 de la Parte 3, compara CP de rango 100 con Tucker-2 de rangos (100, 100, 3, 3). ¿Cuál guarda menos números? Apunta una estimación de la razón.
2. **Ejecuta.** Implementa las dos funciones y calcula los dos conteos.
3. **Explica.** En una frase: ¿qué término hace más grande a Tucker-2 y cómo crece con R?
4. **Comprueba.** Llama a `check_counts(cp_numbers, tucker_numbers)` y compara con tu predicción escrita antes de abrir la solución.

</details>

Prediction / Predicción: ___
Evidence / Evidencia: ___
Revised explanation / Explicación revisada: ___

<details>
<summary>Hint / Pista · if stuck after an attempt / si te atascas tras intentarlo</summary>

`sum(n * r for n, r in zip(shape, ranks) if r < n)` counts only the factors of modes that are actually compressed. The core is `np.prod(ranks)`.

`sum(n * r for n, r in zip(shape, ranks) if r < n)` cuenta solo los factores de los modos que se comprimen de verdad. El núcleo es `np.prod(ranks)`.

</details>

In [ ]:
# TODO 1 / TAREA 1
#
# 1. def cp_numbers(shape, rank): numbers stored by CP at `rank`.
# 2. def tucker_numbers(shape, ranks): core + factors; a mode with
#    ranks[n] == shape[n] stores no factor.
# 3. kernel_shape = (256, 256, 3, 3). Compute CP at rank 100 and
#    Tucker-2 at (100, 100, 3, 3), and their ratio.
#
# Then run: check_counts(cp_numbers, tucker_numbers)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def cp_numbers(shape, rank):
    return rank * sum(shape)


def tucker_numbers(shape, ranks):
    core = int(np.prod(ranks))
    factors = sum(n * r for n, r in zip(shape, ranks) if r < n)
    return core + factors


kernel_shape = (256, 256, 3, 3)
cp_100 = cp_numbers(kernel_shape, 100)
tucker2_100 = tucker_numbers(kernel_shape, (100, 100, 3, 3))
print(f"CP, rank 100:              {cp_100:>8,}")
print(f"Tucker-2, (100, 100, 3, 3): {tucker2_100:>8,}  ({tucker2_100 / cp_100:.1f}× CP)")

check_counts(cp_numbers, tucker_numbers)

In [ ]:
# Visible on purpose: everything below uses these two names.
# Visible a propósito: todo lo que sigue usa estos dos nombres.


def cp_numbers(shape, rank):
    """Numbers CP stores at `rank`: one length-I_n column per mode per term."""
    return rank * sum(shape)


def tucker_numbers(shape, ranks):
    """Core plus factors; a mode kept whole (R_n = I_n) stores no factor."""
    core = int(np.prod(ranks))
    factors = sum(n * r for n, r in zip(shape, ranks) if r < n)
    return core + factors


kernel_shape = (256, 256, 3, 3)
dense_kernel = int(np.prod(kernel_shape))


def first_rank_that_stops_compressing(count):
    """The smallest rank whose count reaches the dense kernel's."""
    return next(r for r in range(1, dense_kernel + 1) if count(r) >= dense_kernel)


print(f"Dense kernel / Kernel denso: {dense_kernel:,} numbers / números")
print("At equal rank, CP costs 518 R and square Tucker-2 costs 512 R + 9 R².")
print("CP stops compressing at rank / CP deja de comprimir en el rango:",
      first_rank_that_stops_compressing(lambda r: cp_numbers(kernel_shape, r)))
print("Square Tucker-2 stops compressing at rank / Tucker-2 cuadrado deja de comprimir en el rango:",
      first_rank_that_stops_compressing(lambda r: tucker_numbers(kernel_shape, (r, r, 3, 3))))

<details>
<summary><strong>What did Exercise 1 show? / ¿Qué mostró el Ejercicio 1?</strong></summary>

At rank 100 Tucker-2 stores **141,200** numbers against CP's **51,800**, about 2.7 times as many. The difference is the core: $9R^2 = 90{,}000$ of Tucker-2's numbers are core entries, one for every pair of an output-channel pattern and an input-channel pattern at every one of the 9 spatial positions. CP pays $518R$, linear in $R$; square Tucker-2 pays $512R + 9R^2$, and the quadratic term takes over almost immediately. CP stops compressing this kernel only at rank 1,139; square Tucker-2 already at rank 230.

**But equal rank is the wrong comparison**, and this is the most common mistake in comparing the two. The two models do not fit the tensor equally well at equal rank — Tucker, with its dense core, fits at least as well, and usually better. The fair question is the one Part 3 asks: *give both the same number of stored numbers, and which fits better?* A count tells you the price of a model; it does not tell you what the model buys.

<br>

**ES.** Con rango 100, Tucker-2 guarda <b>141.200</b> números frente a los <b>51.800</b> de CP, unas 2,7 veces más. La diferencia es el núcleo: 9R² = 90.000 entradas, una por cada par de patrones de canal de salida y de entrada en cada una de las 9 posiciones espaciales. CP paga 518R, lineal en R; Tucker-2 cuadrado paga 512R + 9R², y el término cuadrático manda casi enseguida. <b>Pero comparar a igual rango es un error</b>: a igual rango Tucker ajusta al menos igual de bien. La pregunta justa es la de la Parte 3: <i>con el mismo número de números guardados, ¿cuál ajusta mejor?</i> Un conteo dice el precio de un modelo, no lo que compra.

</details>

## Part 2 · Knowledge graphs: the job is prediction / Grafos de conocimiento: el trabajo es predecir

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

A knowledge graph is a set of facts $(s, r, o)$: *subject*, *relation*, *object* — (Paris, capital of, France). Stack every entity along two axes and every relation along a third and the graph becomes a **binary tensor** $\mathcal{Y}$ with $y_{sro} = 1$ for every recorded fact.

The job is **completion**: score facts nobody recorded, so that a query like *(Paris, located in, ?)* ranks France near the top. A completion model is a factorisation of $\mathcal{Y}$ whose reconstructed entries are the scores.

**The data.** FB15k-237 is the standard benchmark. Bordes et al. (2013) cut FB15k out of Freebase, the crowd-edited knowledge base Google bought and later shut down. Toutanova and Chen (2015) then removed near-duplicate and inverse relations, because many FB15k test facts could be answered just by flipping a training fact. The copy used here is the one distributed with the TuckER paper.

**What a wrong answer costs.** A model that cannot tell a fact from its reversal ranks *(Paris, contains, France)* as high as *(France, contains, Paris)*, and every query downstream inherits the error. So the first question to ask of the data is: **how often is a relation's reversal also a fact?**

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un grafo de conocimiento es un conjunto de hechos (s, r, o): sujeto, relación, objeto. Con las entidades en dos ejes y las relaciones en un tercero, el grafo es un <b>tensor binario</b> con un 1 por cada hecho registrado. El trabajo es <b>completar</b>: puntuar hechos que nadie registró. Un modelo de completado es una factorización de ese tensor cuyas entradas reconstruidas son las puntuaciones.
<br><br>
FB15k-237 es el banco de pruebas estándar: sale de Freebase, y Toutanova y Chen (2015) quitaron relaciones casi duplicadas e inversas porque muchas preguntas de prueba se respondían invirtiendo un hecho de entrenamiento. Un modelo que no distingue un hecho de su inverso puntúa igual <i>(París, contiene, Francia)</i> que <i>(Francia, contiene, París)</i>. La primera pregunta es: <b>¿con qué frecuencia el inverso de un hecho también es un hecho?</b></div>

In [ ]:
# FB15k-237's training split, as distributed with TuckER (Balažević et al. 2019).
# Some lines end in CRLF, so every field is stripped.
raw = fetch(FB15K_TRAIN, "6e4c2782169af21e9743f3b1d200886f5d595bf6bc504ec1351720949c5cdfae")
triples = pd.read_csv(io.BytesIO(raw), sep="\t", header=None,
                      names=["s", "r", "o"], dtype=str)
triples = triples.apply(lambda column: column.str.strip())

n_entities = len(set(triples.s) | set(triples.o))
n_relations = triples.r.nunique()
cells = n_entities * n_relations * n_entities

print(f"Training triples / Tripletas de entrenamiento: {len(triples):,}")
print(f"Entities, relations / Entidades, relaciones: {n_entities:,}, {n_relations}")
print(f"Tensor / Tensor: {n_entities:,} × {n_relations} × {n_entities:,} = {cells:,} cells / celdas")
print(f"Density / Densidad: {len(triples) / cells:.1e}")
print()
print(triples.sample(4, random_state=19).to_string(index=False))

Fifty billion cells, a quarter of a million ones: a density of about $5 \times 10^{-6}$. The entity names are Freebase machine ids like `/m/0f8l9c`; the relation names are paths through Freebase's schema.

That density is the first thing that makes this a *prediction* problem rather than a compression one. Least squares over every cell would treat every missing triple as a known zero, and all but about five in every million of its terms would be those zeros. **Unobserved is not false** — nobody recorded most true facts — so the models below are trained with a ranking or classification loss over observed facts, not a reconstruction loss over the whole tensor.

Now the question about reversals. For each relation, the cell below measures two things:

- **reversed fraction** — the share of its pairs $(s, o)$ whose reversal $(o, s)$ is also a training fact under the same relation;
- **two-role fraction** — the share of its pairs in which the object is also a *subject* of that relation somewhere, or the subject is also an *object*. Only for those pairs can a symmetric score put a reversed pair ahead of a true answer in a ranking query: if no entity ever plays both roles, the reversed pair is never a candidate.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Cincuenta mil millones de celdas y un cuarto de millón de unos: densidad de unos 5 × 10⁻⁶. Por eso esto es un problema de <i>predicción</i> y no de compresión: mínimos cuadrados sobre todas las celdas trataría cada tripleta ausente como un cero conocido. <b>No observado no es falso</b>, así que los modelos se entrenan con pérdidas de clasificación o de ranking sobre hechos observados.
<br><br>
La celda siguiente mide, por relación, la <b>fracción invertida</b> (pares cuyo inverso también es un hecho) y la <b>fracción de doble papel</b> (pares donde el objeto también aparece como sujeto de esa relación, o el sujeto como objeto). Solo en esos pares una puntuación simétrica puede colocar un par invertido por delante de una respuesta verdadera.</div>

In [ ]:
pairs = triples.drop_duplicates()

# A pair is reversed when (o, r, s) is also a fact. Merge the table with its own mirror.
# Un par está invertido si (o, r, s) también es un hecho: cruza la tabla con su espejo.
mirror = pairs.rename(columns={"s": "o", "o": "s"}).assign(reversed=True)
pairs = pairs.merge(mirror, on=["s", "r", "o"], how="left")
pairs["reversed"] = pairs["reversed"].eq(True) & (pairs.s != pairs.o)

# A pair is exposed when its object is also a subject of r, or its subject an object.
subjects = pairs[["r", "s"]].drop_duplicates().rename(columns={"s": "o"}).assign(o_is_subject=True)
objects = pairs[["r", "o"]].drop_duplicates().rename(columns={"o": "s"}).assign(s_is_object=True)
pairs = pairs.merge(subjects, on=["r", "o"], how="left").merge(objects, on=["r", "s"], how="left")
pairs["two_role"] = pairs.o_is_subject.eq(True) | pairs.s_is_object.eq(True)

relations = pairs.groupby("r").agg(
    train_pairs=("s", "size"),
    reversed_fraction=("reversed", "mean"),
    two_role_fraction=("two_role", "mean"),
)
total = relations.train_pairs.sum()
never = relations.reversed_fraction == 0
mostly = relations.reversed_fraction >= 0.5
between = (relations.reversed_fraction > 0.1) & (relations.reversed_fraction < 0.5)
directional = relations[never]
exposed = (directional.train_pairs * directional.two_role_fraction).sum() / directional.train_pairs.sum()
contains = relations.loc["/location/location/contains"]

print(f"Never reversed / Nunca invertidas: {never.sum()} relations carrying "
      f"{relations.train_pairs[never].sum() / total:.0%} of triples")
print(f"Reversed for at least half their pairs / Invertidas en al menos la mitad: {mostly.sum()}")
print(f"Between 10% and 50% / Entre el 10% y el 50%: {between.sum()}")
print(f"Of the never-reversed, no entity plays both roles in / "
      f"De las nunca invertidas, ninguna entidad tiene doble papel en: {(directional.two_role_fraction == 0).sum()}")
print(f"Exposed pairs in never-reversed relations / Pares expuestos en relaciones nunca invertidas: {exposed:.1%}")
print(f"/location/location/contains: {int(contains.train_pairs):,} pairs, "
      f"{contains.reversed_fraction:.0%} reversed, {contains.two_role_fraction:.0%} exposed")

fig, ax = plt.subplots(figsize=(7.5, 3.0))
ax.hist(relations.reversed_fraction, bins=np.linspace(0, 1, 21),
        weights=relations.train_pairs / total, color=ACCENT, edgecolor="white")
ax.set_xlabel("fraction of a relation's pairs whose reversal is also recorded")
ax.set_ylabel("share of training triples")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("FB15k-237: relations are either directional or symmetric")
plt.tight_layout()
plt.show()

Relations split cleanly into two kinds. Most — carrying about four in five training triples — are **directional**: their reversal is never recorded. Two dozen are **symmetric**: siblings, cast members of the same season, combatants in the same war. Only one sits between 10% and 50%. A relation is almost never partly both.

The directional ones are where a symmetric model must be wrong. But the two-role column says how often it gets the *chance*: in most directional relations no entity is ever both a subject and an object, and across all of them only about 3% of pairs are exposed. The exception is a **hierarchy**. A city contains districts and is contained by a country, so in `/location/location/contains` four pairs in five are exposed.

Keep that number in mind. It is going to explain why a model with a provably wrong symmetry does surprisingly well on this benchmark.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Las relaciones se dividen limpiamente: la mayoría —con unas cuatro de cada cinco tripletas— son <b>direccionales</b> (su inverso nunca aparece) y un par de docenas son <b>simétricas</b> (hermanos, reparto de una misma temporada). Solo una queda entre el 10% y el 50%.
<br><br>
En las direccionales un modelo simétrico tiene que equivocarse, pero la columna de doble papel dice cuántas veces tiene <i>ocasión</i>: en la mayoría ninguna entidad es a la vez sujeto y objeto, y en total solo un 3% de los pares está expuesto. La excepción es una <b>jerarquía</b>: una ciudad contiene distritos y está contenida en un país, así que en <code>contains</code> cuatro de cada cinco pares están expuestos. Ese número explicará por qué un modelo con una simetría demostrablemente errónea funciona sorprendentemente bien en este banco de pruebas.</div>

### Scoring functions are cores / Las funciones de puntuación son núcleos

Every model in this family scores a triple as a **multilinear form** in three embeddings: a vector $e_s$ for the subject, $w_r$ for the relation, $e_o$ for the object. Written as a factorisation of $\mathcal{Y}$, the entity table is the factor of modes 1 and 3, the relation table is the factor of mode 2, and **the core decides which coordinates may meet**.

- **DistMult** (Yang et al. 2015). Superdiagonal core; one entity table used for both subject and object ("tied").
$$
\phi(s, r, o) = \sum_k e_{s,k}\, w_{r,k}\, e_{o,k} = \phi(o, r, s).
$$
The sum is unchanged when $s$ and $o$ swap, **for every relation and every embedding** — that is the predict-first cell.

- **CP with untied entities.** Superdiagonal core, separate subject table $A$ and object table $C$: $\phi = \sum_k a_{s,k}\, w_{r,k}\, c_{o,k}$. Asymmetric — but $a_s$ and $c_s$ share nothing, so what the model learns about Paris as a subject says nothing about Paris as an object. Lacroix et al. (2018) fix that by also training on the reciprocal triple $(o, r^{-1}, s)$.

- **ComplEx** (Trouillon et al. 2016). Superdiagonal core over *complex* embeddings, tied: $\phi = \operatorname{Re} \sum_k e_{s,k}\, w_{r,k}\, \overline{e_{o,k}}$. The complex conjugate on the object breaks the symmetry.

- **RESCAL** (Nickel et al. 2011). A full $d_e \times d_e$ matrix $M_r$ per relation: $\phi = e_s^\top M_r\, e_o$. That is Tucker-2: the two entity modes compressed, the relation mode left at full length, and the core is the stack of all the $M_r$.

- **TuckER** (Balažević et al. 2019). A dense core $\mathcal{G} \in \mathbb{R}^{d_e \times d_r \times d_e}$ shared by every relation, tied entities:
$$
\phi(s, r, o) = \mathcal{G} \times_1 e_s \times_2 w_r \times_3 e_o = \sum_{a,b,c} g_{abc}\, e_{s,a}\, w_{r,b}\, e_{o,c} .
$$
Each relation's matrix, $M_r = \mathcal{G} \times_2 w_r$, is a mixture of $d_r$ shared slices. Nothing forces a slice to be symmetric, so the asymmetry comes from the core, while entities keep one vector each.

TuckER's paper shows that RESCAL, DistMult, ComplEx and SimplE are all special cases of TuckER — each is a TuckER core with a constraint. **The modelling choice is which constraint to pay for.** The cell below measures the one constraint that matters here, symmetry, on random embeddings. Exercise 2 then makes the "special case" claim concrete.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Todos estos modelos puntúan una tripleta como una <b>forma multilineal</b> en tres embeddings: e_s, w_r y e_o. Como factorización del tensor, la tabla de entidades es el factor de los modos 1 y 3, la de relaciones el del modo 2, y <b>el núcleo decide qué coordenadas pueden encontrarse</b>.
<br><br>
<b>DistMult</b>: núcleo superdiagonal y una sola tabla de entidades; es simétrico en s y o para toda relación y todo embedding. <b>CP con entidades separadas</b>: asimétrico, pero lo que aprende de París como sujeto no dice nada de París como objeto (Lacroix et al. lo arreglan entrenando también con la tripleta recíproca). <b>ComplEx</b>: núcleo superdiagonal sobre embeddings complejos; el conjugado rompe la simetría. <b>RESCAL</b>: una matriz M_r completa por relación, es decir, Tucker-2. <b>TuckER</b>: un núcleo denso compartido por todas las relaciones; cada M_r es una mezcla de rebanadas compartidas y nada obliga a que sean simétricas.
<br><br>
RESCAL, DistMult y ComplEx son casos particulares de TuckER: cada uno es un núcleo de TuckER con una restricción. <b>La elección de modelo es qué restricción pagar.</b></div>

In [ ]:
# Random embeddings for one subject, one object and one relation, d = 6.
# Embeddings aleatorios para un sujeto, un objeto y una relación, d = 6.
d = 6
e_s, e_o, w_r = rng.standard_normal((3, d))
a_s, c_s, a_o, c_o = rng.standard_normal((4, d))                  # untied tables
z_s, z_o, z_r = rng.standard_normal((3, d)) + 1j * rng.standard_normal((3, d))
M_r = rng.standard_normal((d, d))                                  # RESCAL's matrix
G = rng.standard_normal((d, d, d))                                 # TuckER's core (d_r = d)

scores = {
    "DistMult":  (np.einsum("k,k,k->", e_s, w_r, e_o), np.einsum("k,k,k->", e_o, w_r, e_s)),
    "CP, untied": (np.einsum("k,k,k->", a_s, w_r, c_o), np.einsum("k,k,k->", a_o, w_r, c_s)),
    "ComplEx":   (np.real(np.sum(z_s * z_r * np.conj(z_o))), np.real(np.sum(z_o * z_r * np.conj(z_s)))),
    "RESCAL":    (e_s @ M_r @ e_o, e_o @ M_r @ e_s),
    "TuckER":    (np.einsum("abc,a,b,c->", G, e_s, w_r, e_o), np.einsum("abc,a,b,c->", G, e_o, w_r, e_s)),
}
print(f"{'model':<12}{'φ(s, r, o)':>12}{'φ(o, r, s)':>12}   symmetric? / ¿simétrico?")
for name, (forward, backward) in scores.items():
    print(f"{name:<12}{forward:>12.4f}{backward:>12.4f}   {np.isclose(forward, backward)}")

In [ ]:
# Checker for Exercise 2 / Comprobador del Ejercicio 2.


def check_tucker_score(tucker_score, distmult_core, rescal_core):
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    g = np.random.default_rng(0)
    d, d_r, n_rel = 5, 2, 3                     # entity size, relation size, relations
    E = g.standard_normal((4, d))
    W = g.standard_normal((n_rel, d_r))
    core = g.standard_normal((d, d_r, d))
    M_stack = g.standard_normal((n_rel, d, d))
    try:
        want = sum(core[a, b, c] * E[0, a] * W[1, b] * E[2, c]
                   for a in range(d) for b in range(d_r) for c in range(d))
        got = tucker_score(core, E[0], W[1], E[2])
        ok(np.isclose(got, want),
           "tucker_score contracts the core with e_s, w_r and e_o on modes 0, 1 and 2.",
           f"tucker_score returned {got:.4f}; the triple sum over a, b, c gives {want:.4f}. "
           "Check which einsum index goes with which vector.")
        G_dm = distmult_core(d)
        ok(np.shape(G_dm) == (d, d, d),
           f"distmult_core({d}) has shape {(d, d, d)}.",
           f"distmult_core({d}) has shape {np.shape(G_dm)}; with d_r = d_e = {d} it is {(d, d, d)}.")
        w = g.standard_normal(d)
        want = np.sum(E[0] * w * E[2])
        got = tucker_score(G_dm, E[0], w, E[2])
        ok(np.isclose(got, want),
           "With the superdiagonal core, TuckER's score is DistMult's.",
           f"With your core the score is {got:.4f}; DistMult gives {want:.4f}. "
           "Only g[k, k, k] should be non-zero.")
        G_rs = rescal_core(M_stack)
        onehot = np.eye(n_rel)[2]
        want = E[1] @ M_stack[2] @ E[3]
        got = tucker_score(G_rs, E[1], onehot, E[3])
        ok(np.isclose(got, want),
           "With the stacked matrices as core and a one-hot w_r, TuckER's score is RESCAL's.",
           f"With your core and relation 2 one-hot the score is {got:.4f}; RESCAL gives "
           f"{want:.4f}. The relation must index the core's *middle* mode.")
    except Exception as error:  # a learner's function can fail in any way
        notes.append(f"❌ Your function raised {type(error).__name__}: {error}")
    print("\n".join(notes))

### Exercise 2 · Two models as one core / Ejercicio 2 · Dos modelos como un núcleo

Make TuckER's claim concrete: DistMult and RESCAL are TuckER with a particular core.

1. `tucker_score(G, e_s, w_r, e_o)` — TuckER's score, $\mathcal{G} \times_1 e_s \times_2 w_r \times_3 e_o$, as one `np.einsum`.
2. `distmult_core(d)` — a $d \times d \times d$ core that makes `tucker_score` equal DistMult's score whenever $w_r$ has length $d$.
3. `rescal_core(M_stack)` — given RESCAL's matrices stacked as `M_stack` of shape `(n_rel, d, d)`, a core that makes `tucker_score(G, e_s, onehot_r, e_o)` equal $e_s^\top M_r e_o$, where `onehot_r` is the one-hot vector of relation $r$.

**Before you code**, answer on paper: DistMult's core has $d^3$ entries but only $d$ of them are non-zero. How many free parameters does TuckER's core have at $d_e = d_r = 200$, and how does that compare with one entity's embedding?

Then run `check_tucker_score(tucker_score, distmult_core, rescal_core)`.

<details>
<summary>Español</summary>

Concreta la afirmación de TuckER: DistMult y RESCAL son TuckER con un núcleo concreto. (1) `tucker_score(G, e_s, w_r, e_o)` con un solo `np.einsum`; (2) `distmult_core(d)`, un núcleo d × d × d que reproduce DistMult; (3) `rescal_core(M_stack)`, un núcleo que con `w_r` one-hot reproduce RESCAL. <b>Antes de programar</b>: ¿cuántos parámetros libres tiene el núcleo de TuckER con d_e = d_r = 200, y cómo se compara con el embedding de una entidad? Después ejecuta `check_tucker_score(...)`.

</details>

<details>
<summary>Hint / Pista</summary>

For DistMult, `g[k, k, k] = 1` and zero elsewhere — the same superdiagonal core as Part 1. For RESCAL, the relation must index the core's middle mode, so the core is `M_stack` with its first two axes swapped.

Para DistMult, `g[k, k, k] = 1` y cero en el resto: el mismo núcleo superdiagonal de la Parte 1. Para RESCAL, la relación indexa el modo central del núcleo, así que el núcleo es `M_stack` con los dos primeros ejes intercambiados.

</details>

In [ ]:
# TODO 2 / TAREA 2
#
# 1. def tucker_score(G, e_s, w_r, e_o): one np.einsum.
# 2. def distmult_core(d): a (d, d, d) core with g[k, k, k] = 1.
# 3. def rescal_core(M_stack): (n_rel, d, d) -> a (d, n_rel, d) core.
#
# Then run: check_tucker_score(tucker_score, distmult_core, rescal_core)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def tucker_score(G, e_s, w_r, e_o):
    return np.einsum("abc,a,b,c->", G, e_s, w_r, e_o)


def distmult_core(d):
    core = np.zeros((d, d, d))
    core[np.arange(d), np.arange(d), np.arange(d)] = 1.0
    return core


def rescal_core(M_stack):
    return np.transpose(M_stack, (1, 0, 2))


check_tucker_score(tucker_score, distmult_core, rescal_core)

In [ ]:
# Visible on purpose: the toy world below uses tucker_score's einsum.
# Visible a propósito: el mundo de juguete de abajo usa el einsum de tucker_score.


def tucker_score(G, e_s, w_r, e_o):
    return np.einsum("abc,a,b,c->", G, e_s, w_r, e_o)


print("TuckER core at d_e = d_r = 200 / Núcleo de TuckER con d_e = d_r = 200:",
      f"{200 ** 3:,} numbers, shared by every relation; one entity costs 200.")

<details>
<summary><strong>What did Exercise 2 show? / ¿Qué mostró el Ejercicio 2?</strong></summary>

The three models are one formula with three cores. DistMult's core is the superdiagonal of Part 1: coordinate $k$ of the subject meets only coordinate $k$ of the relation and of the object. RESCAL's is the stack of per-relation matrices, placed so the relation indexes the middle mode. TuckER leaves the core free.

The paper-and-pencil question has a surprising answer. TuckER's core at $d_e = d_r = 200$ has **8,000,000** free numbers, forty thousand times one entity's embedding. That sounds expensive — until you notice the core is paid for **once**, for the whole graph, while entity embeddings are paid for once *per entity*. On a graph with $10^5$ entities, that trade decides the parameter count, as the published table below shows.

<br>

**ES.** Los tres modelos son una fórmula con tres núcleos: DistMult usa la superdiagonal (la coordenada k solo se encuentra con la k), RESCAL la pila de matrices por relación, y TuckER deja el núcleo libre. El núcleo de TuckER con d_e = d_r = 200 tiene <b>8.000.000</b> de números, cuarenta mil veces un embedding de entidad; pero se paga <b>una vez</b> para todo el grafo, mientras que los embeddings se pagan una vez <i>por entidad</i>. Con 10⁵ entidades, ese intercambio decide el número de parámetros.

</details>

### Does symmetry cost a fact? A world small enough to check by hand / ¿Cuesta un hecho la simetría? Un mundo que se comprueba a mano

FB15k-237 is too big to train here in a few minutes, and a benchmark number would hide the mechanism anyway. So build a world with exactly the structure that exposed `contains`: **3 countries, each containing 3 cities, each containing 3 districts** — 39 entities. Two relations:

- `contains`, directional, 36 facts, with cities in **both roles** (contained by a country, containing districts) — the exposed case;
- `neighbour_of`, symmetric: cities in the same country, 18 facts.

**The loss.** Both models are trained the way TuckER was: **1-N scoring with binary cross-entropy**. For each pair $(s, r)$, score *every* entity as a candidate object at once, and push the recorded objects towards 1 and every other entity towards 0 with a sigmoid and BCE. On a real graph you would add label smoothing (TuckER uses 0.1), because a 0 there means "unrecorded", not "false"; in this toy world the list of facts is complete, so a 0 really is false and smoothing is left out.

**What to measure.** Not accuracy on the whole tensor — a model can be right about almost every cell and still wrong about the one thing that matters. Instead:

1. For each true fact $(s, \text{contains}, o)$, is it scored **strictly above** its reversal $(o, \text{contains}, s)$?
2. For each city — the entities with both roles — does the query *(city, contains, ?)* rank its own **country** at or above one of its true **districts**?

The two models get comparable budgets: 8-dimensional entity embeddings for both; TuckER adds a 8 × 4 × 8 core.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>FB15k-237 es demasiado grande para entrenarlo aquí, y una cifra de benchmark escondería el mecanismo. Así que construimos un mundo con la estructura exacta que expone <code>contains</code>: <b>3 países, cada uno con 3 ciudades, cada una con 3 distritos</b> (39 entidades), con <code>contains</code> (direccional, 36 hechos, las ciudades en ambos papeles) y <code>neighbour_of</code> (simétrica, 18 hechos).
<br><br>
Ambos modelos se entrenan como TuckER: <b>puntuación 1-N con entropía cruzada binaria</b>, puntuando todas las entidades a la vez como objeto candidato. En un grafo real se añadiría suavizado de etiquetas, porque un 0 significa «no registrado», no «falso»; aquí la lista de hechos es completa y no hace falta.
<br><br>
No medimos la exactitud sobre todo el tensor, sino: (1) si cada hecho verdadero de <code>contains</code> puntúa <b>estrictamente por encima</b> de su inverso, y (2) si, para cada ciudad, la consulta <i>(ciudad, contains, ?)</i> coloca a su <b>país</b> a la altura o por encima de alguno de sus <b>distritos</b>.</div>

In [ ]:
# The toy world / El mundo de juguete.
names, contains_pairs = [], []
for c in range(3):
    country = len(names); names.append(f"country{c}")
    for k in range(3):
        city = len(names); names.append(f"city{c}{k}"); contains_pairs.append((country, city))
        for t in range(3):
            district = len(names); names.append(f"district{c}{k}{t}"); contains_pairs.append((city, district))
cities = [i for i, name in enumerate(names) if name.startswith("city")]
country_of = {city: parent for parent, city in contains_pairs if city in cities}
neighbour_pairs = [(a, b) for a in cities for b in cities if a != b and country_of[a] == country_of[b]]

n_toy, CONTAINS, NEIGHBOUR = len(names), 0, 1
Y = torch.zeros(n_toy, 2, n_toy)                          # subject × relation × object
for s, o in contains_pairs:
    Y[s, CONTAINS, o] = 1
for s, o in neighbour_pairs:
    Y[s, NEIGHBOUR, o] = 1


def train_kg(model, d_e=8, d_r=4, epochs=600, seed=0):
    """1-N scoring with BCE, full batch. Returns every score φ(s, r, o) and the loss."""
    g = torch.Generator().manual_seed(seed)
    E = nn.Parameter(0.3 * torch.randn(n_toy, d_e, generator=g))
    if model == "DistMult":
        W = nn.Parameter(0.3 * torch.randn(2, d_e, generator=g))
        params = [E, W]

        def all_scores():
            return torch.einsum("sk,rk,ok->sro", E, W, E)
    else:
        W = nn.Parameter(0.3 * torch.randn(2, d_r, generator=g))
        G = nn.Parameter(0.3 * torch.randn(d_e, d_r, d_e, generator=g))
        params = [E, W, G]

        def all_scores():
            return torch.einsum("abc,sa,rb,oc->sro", G, E, W, E)

    optimiser = torch.optim.Adam(params, lr=0.05)
    for _ in range(epochs):
        optimiser.zero_grad()
        loss = F.binary_cross_entropy_with_logits(all_scores(), Y)
        loss.backward()
        optimiser.step()
    with torch.no_grad():
        return all_scores().numpy(), loss.item(), sum(p.numel() for p in params)


print(f"{'':10}{'numbers':>8}{'loss':>9}   true fact above its reversal   country ≥ a district")
toy, separated = {}, {}
for model in ("DistMult", "TuckER"):
    S, loss, count = train_kg(model)
    toy[model] = S
    separated[model] = sum(S[s, CONTAINS, o] > S[o, CONTAINS, s] + 1e-4 for s, o in contains_pairs)
    country_high = sum(
        S[c, CONTAINS, country_of[c]] >= min(S[c, CONTAINS, o] for p, o in contains_pairs if p == c)
        for c in cities)
    print(f"{model:10}{count:>8}{loss:>9.4f}   {separated[model]:>2} of {len(contains_pairs)}"
          f"{'':>22}{country_high} of {len(cities)} cities")
print()
print(f"Facts scored above their reversal / Hechos por encima de su inverso: "
      f"DistMult {separated['DistMult']}, TuckER {separated['TuckER']} (of {len(contains_pairs)})")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


for model, S in toy.items():
    print(f"{model}: P(city00 contains country0) = {sigmoid(S[1, CONTAINS, 0]):.2f}, "
          f"P(country0 contains city00) = {sigmoid(S[0, CONTAINS, 1]):.2f}")
print(f"DistMult's largest |φ(s,r,o) − φ(o,r,s)| over all cells: "
      f"{np.abs(toy['DistMult'] - toy['DistMult'].transpose(2, 1, 0)).max():.1e}")

DistMult separates **none** of the 36 `contains` facts from their reversals — the largest difference between any score and its mirror is float32 rounding. Faced with *(country0, contains, city00)* = true and *(city00, contains, country0)* = false, and forced to give them one score, it gives both a probability near one half: the loss-minimising compromise between a 1 and a 0. And for most cities the country ranks alongside the true districts, because DistMult cannot put it anywhere else. TuckER, with a dense core, gets every fact right and drives the reversal to near zero.

That is the mechanism. Now the surprise: on FB15k-237 this failure costs DistMult very little, and the reason is the two-role column you computed. Only about 3% of pairs in directional relations can be hurt, so a model that is provably wrong on hierarchies is barely wrong on the benchmark.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>DistMult no separa <b>ninguno</b> de los 36 hechos de <code>contains</code> de su inverso; la mayor diferencia entre una puntuación y su espejo es redondeo de float32. Obligado a dar la misma puntuación a un 1 y a un 0, les da a ambos una probabilidad cercana a un medio. Y en la mayoría de las ciudades el país queda a la altura de los distritos verdaderos. TuckER, con núcleo denso, acierta todos y lleva el inverso casi a cero.
<br><br>
La sorpresa: en FB15k-237 este fallo le cuesta muy poco a DistMult, y la razón es la columna de doble papel que calculaste: solo un 3% de los pares de relaciones direccionales puede verse afectado.</div>

In [ ]:
# Published FB15k-237 results, and what each model costs at its published size.
# Resultados publicados en FB15k-237 y lo que cuesta cada modelo con su tamaño publicado.
ENTITIES_ALL = 14_541          # entities in train + valid + test / en los tres conjuntos
RECIPROCAL = 2 * 237           # both papers also train on reciprocal relations

sizes = {  # per entity, per relation, core
    "CP-N3, rank 4000":      (2 * 4000, 4000, 0),       # separate subject and object tables
    "ComplEx-N3, rank 2000": (2 * 2000, 2 * 2000, 0),   # real and imaginary parts
    "TuckER, d_e = d_r = 200": (200, 200, 200 ** 3),
}
params = {name: ENTITIES_ALL * e + RECIPROCAL * r + core for name, (e, r, core) in sizes.items()}

published = pd.DataFrame([
    ("DistMult",   "superdiagonal",          "tied",   "",      0.343, "Ruffinelli et al. 2020"),
    ("ComplEx",    "superdiagonal, complex", "tied",   "",      0.348, "Ruffinelli et al. 2020"),
    ("RESCAL",     "full matrix / relation", "tied",   "",      0.356, "Ruffinelli et al. 2020"),
    ("CP-N3",      "superdiagonal",          "untied", params["CP-N3, rank 4000"], 0.36, "Lacroix et al. 2018"),
    ("ComplEx-N3", "superdiagonal, complex", "tied",   params["ComplEx-N3, rank 2000"], 0.37, "Lacroix et al. 2018"),
    ("TuckER",     "dense, shared",          "tied",   params["TuckER, d_e = d_r = 200"], 0.358, "Balažević et al. 2019"),
], columns=["model", "core", "entities", "parameters", "MRR", "source"])
published["parameters"] = [f"{p / 1e6:.1f}M" if p else "" for p in published.parameters]
print(published.to_string(index=False))
cp_n3, complex_n3, tucker_kg = params.values()
print()
print(f"TuckER matches CP-N3 with {cp_n3 / tucker_kg:.0f}× fewer parameters, "
      f"and ComplEx-N3 with {complex_n3 / tucker_kg:.1f}× fewer.")
print(f"At 10^5 entities: TuckER ≈ {1e5 * 200 + 200 ** 3:.0e} numbers, CP-N3 ≈ {1e5 * 8000:.0e}.")

MRR is the mean reciprocal rank of the true entity over test queries; higher is better, 1 is perfect. The three Ruffinelli et al. rows come from one common hyperparameter search in LibKGE, where embedding sizes vary by model, so they carry no count.

Four readings, each one a claim you should now be able to defend:

- **Symmetry costs less than the algebra suggests.** DistMult reaches 0.343 once properly tuned. TuckER's own paper lists it at 0.241 — so the *training recipe* moved one model by about 0.10 MRR, more than the whole spread between all six rows. Your exposure count says why: on this benchmark the symmetry rarely gets the chance to hurt.
- **Untied CP is not symmetric, and ties TuckER.** CP-N3 reaches 0.36 against TuckER's 0.358.
- **Tucker's case here is size.** At the published settings TuckER matches CP-N3 with about 11× fewer parameters. CP-N3's count is almost all entity tables (8,000 numbers per entity); TuckER's is mostly its $200^3$ core, which **does not grow with the graph**.
- **The relation dimension is a separate dial.** TuckER can set $d_r \ne d_e$ (it uses $d_r = 30$ on WN18RR, which has 11 relations). CP and ComplEx force one rank on every mode.

The ranking this supports: rule out DistMult when entities play both roles in an asymmetric relation — containment, parenthood, citation. Choose a Tucker core when entity count or memory is the constraint. Treat the MRR gaps between CP-N3, ComplEx-N3 and TuckER as noise next to the training recipe.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>MRR es el rango recíproco medio de la entidad verdadera; más alto es mejor. Cuatro lecturas: (1) <b>la simetría cuesta menos de lo que sugiere el álgebra</b>: DistMult bien ajustado llega a 0,343, y el propio artículo de TuckER lo daba en 0,241, así que la <i>receta de entrenamiento</i> movió un modelo más que toda la diferencia entre las seis filas; (2) <b>CP sin atar no es simétrico y empata con TuckER</b>; (3) <b>el argumento de Tucker aquí es el tamaño</b>: TuckER iguala a CP-N3 con unas 11 veces menos parámetros, porque su núcleo no crece con el grafo; (4) <b>la dimensión de relación es un mando aparte</b>.
<br><br>
Conclusión: descarta DistMult cuando las entidades tienen doble papel en relaciones asimétricas (contención, parentesco, citas); elige un núcleo de Tucker cuando la memoria o el número de entidades mandan; y trata las diferencias de MRR entre CP-N3, ComplEx-N3 y TuckER como ruido frente a la receta de entrenamiento.</div>

## Part 3 · Compressing a convolution: the job is storage / Comprimir una convolución: el trabajo es guardar

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

A 2-D convolution layer's weights form a 4-way tensor $\mathcal{K}$ of shape *out-channels × in-channels × height × width*. The layer computes, for output channel $t$ at pixel $(p, q)$,

$$
y_t(p, q) = \sum_{s} \sum_{i, j} \mathcal{K}_{t, s, i, j}\; x_s(p + i, q + j) .
$$

(PyTorch's `conv2d` is a cross-correlation, which is what this formula is; the indices $i, j$ run over the kernel window around the pixel.) Here the job is the opposite of Part 2's: nothing is hidden, the whole tensor is known, and the question is **how few numbers can reproduce it well enough**.

**The data.** `layer3.1.conv2.weight` from ResNet-18 (He et al. 2016), in the ImageNet weights torchvision ships as `IMAGENET1K_V1`: 256 × 256 × 3 × 3, stride 1, padding 1. A trained kernel is the right test because it carries whatever structure training gave it; a random tensor of the same shape would have none. The file here is the one the blog post extracted, pinned by its hash.

**What a wrong answer costs.** A factorisation that fits the weights badly needs long fine-tuning to recover accuracy. One that fits them *unstably* may not fine-tune at all. Keep the second half of that sentence in mind.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Los pesos de una capa de convolución 2-D forman un tensor de 4 modos: canales de salida × canales de entrada × alto × ancho. Aquí el trabajo es el contrario al de la Parte 2: no hay nada oculto, el tensor entero se conoce, y la pregunta es <b>con cuántos números se puede reproducir lo bastante bien</b>.
<br><br>
Los datos: <code>layer3.1.conv2.weight</code> de ResNet-18 con los pesos de ImageNet de torchvision, 256 × 256 × 3 × 3. Un kernel entrenado es la prueba correcta porque lleva la estructura que le dio el entrenamiento. Una factorización que ajusta mal los pesos necesita mucho reajuste; una que los ajusta de forma <i>inestable</i> puede no reajustarse nunca.</div>

In [ ]:
KERNEL_SHA256 = "2ace0a79d66e0d1ed2239639269df7f831122100033483a41779b760202e7df5"
kernel = np.load(io.BytesIO(fetch(
    BLOG_DATA + "resnet18_layer3_1_conv2.npy", KERNEL_SHA256,
    copy="https://project-delphi.github.io/tensors-workshop/data/resnet18_layer3_1_conv2.npy")))
kernel = kernel.astype(np.float64)
C_out, C_in, kh, kw = kernel.shape
kernel_norm = np.linalg.norm(kernel)

print(f"Kernel / Kernel: {kernel.shape}, {kernel.size:,} weights / pesos")
print(f"‖K‖ = {kernel_norm:.3f}; largest |weight| / mayor |peso| = {np.abs(kernel).max():.3f}")

### CP is four convolutions / CP son cuatro convoluciones

Write the rank-$R$ CP of the kernel with one factor per mode, $U^{\text{out}} \in \mathbb{R}^{256 \times R}$, $U^{\text{in}} \in \mathbb{R}^{256 \times R}$, $U^{h}, U^{w} \in \mathbb{R}^{3 \times R}$:

$$
\mathcal{K}_{t,s,i,j} \approx \sum_{r=1}^{R} U^{\text{out}}_{t r}\, U^{\text{in}}_{s r}\, U^{h}_{i r}\, U^{w}_{j r} .
$$

Substitute it into the convolution and move each sum as far inward as it will go:

$$
y_t(p, q) \approx \sum_{r} U^{\text{out}}_{t r} \sum_{i} U^{h}_{i r} \sum_{j} U^{w}_{j r} \underbrace{\sum_{s} U^{\text{in}}_{s r}\, x_s(p+i, q+j)}_{z_r(p+i,\, q+j)} .
$$

Read it from the inside out and it is four layers (Lebedev et al. 2015):

1. a **1×1 conv**, 256 → $R$ channels, weights $U^{\text{in}}$ — the innermost sum, $z_r$;
2. a **3×1 depthwise conv** on each of the $R$ channels, weights $U^{h}$ — the sum over $i$;
3. a **1×3 depthwise conv** on each channel, weights $U^{w}$ — the sum over $j$;
4. a **1×1 conv**, $R$ → 256 channels, weights $U^{\text{out}}$ — the outer sum.

*Depthwise* means each channel is convolved on its own (`groups=R` in PyTorch): a rank-one term never mixes with another, which is the superdiagonal core again. The chain stores $R(256 + 3 + 3 + 256) = 518R$ weights, and at stride 1 it does that many multiply-adds per output pixel.

The next cell fits CP at $R = 36$ — the rank that stores **32 times fewer** numbers than the dense kernel — with TensorLy's ALS and line search, for 100 iterations (about 20 s on Colab). It also records one number you have not met yet: the Frobenius norm of the **largest single rank-one term**, divided by the kernel's own norm. Part 3 will explain why that is worth watching.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Sustituye la CP de rango R del kernel en la convolución y mete cada suma todo lo posible hacia dentro: sale una cadena de <b>cuatro capas</b> (Lebedev et al. 2015): (1) conv 1×1 de 256 a R canales con U<sup>in</sup>; (2) conv 3×1 <i>depthwise</i> por canal con U<sup>h</sup>; (3) conv 1×3 depthwise con U<sup>w</sup>; (4) conv 1×1 de R a 256 canales con U<sup>out</sup>. <i>Depthwise</i> significa que cada canal se convoluciona por separado: un término de rango uno nunca se mezcla con otro, que es otra vez el núcleo superdiagonal. La cadena guarda 518R pesos.
<br><br>
La celda siguiente ajusta CP con R = 36, el rango que guarda <b>32 veces menos</b> números que el kernel denso, y registra un número nuevo: la norma del <b>mayor término de rango uno</b> dividida por la del kernel.</div>

In [ ]:
R_CP = 36                                   # 518 × 36 = 18,648 numbers: 32× compression
start = time.time()
cp32 = parafac(kernel, rank=R_CP, init="random", random_state=0,
               n_iter_max=100, tol=1e-10, linesearch=True)
cp32_error = rel_error(kernel, tl.cp_to_tensor(cp32))

# ‖a_r ∘ b_r ∘ c_r ∘ d_r‖ = |λ_r| · Π_n ‖column r of factor n‖
term_norms = np.abs(cp32.weights) * np.prod([np.linalg.norm(f, axis=0) for f in cp32.factors], axis=0)
cp32_largest = term_norms.max() / kernel_norm

print(f"CP rank {R_CP}: {cp_numbers(kernel.shape, R_CP):,} numbers, "
      f"{kernel.size / cp_numbers(kernel.shape, R_CP):.0f}× compression "
      f"({time.time() - start:.0f} s)")
print(f"Relative error / Error relativo: {cp32_error:.3f}")
print(f"Largest rank-one term / Mayor término de rango uno: {cp32_largest:.2f} × ‖K‖")

In [ ]:
# Checker for Exercise 3 / Comprobador del Ejercicio 3.


def check_cp_convs(build_cp_convs):
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    try:
        chain = build_cp_convs(cp32)
        x = torch.randn(4, C_in, 14, 14, generator=torch.Generator().manual_seed(0))
        K_cp = t32(tl.cp_to_tensor(cp32))
        with torch.no_grad():
            reference = F.conv2d(x, K_cp, padding=1)
            got = chain(x)
        ok(tuple(got.shape) == tuple(reference.shape),
           f"The chain maps {tuple(x.shape)} to {tuple(got.shape)}, like the 3×3 conv.",
           f"The chain outputs {tuple(got.shape)}; the 3×3 conv outputs "
           f"{tuple(reference.shape)}. Check the padding of the two depthwise layers.")
        if tuple(got.shape) == tuple(reference.shape):
            gap = (got - reference).abs().max().item()
            scale = reference.abs().max().item()
            ok(gap < 1e-4 * scale,
               f"Largest output difference {gap:.1e} on outputs up to {scale:.1f}: float32 rounding.",
               f"Largest output difference {gap:.2g} on outputs up to {scale:.2g}. Check which "
               "factor goes in which layer, the transposes, and that the CP weights are folded in.")
        count = sum(p.numel() for p in chain.parameters())
        ok(count == cp_numbers(kernel.shape, R_CP),
           f"The chain holds {count:,} weights, the formula's 518 × {R_CP}.",
           f"The chain holds {count:,} weights; CP stores {cp_numbers(kernel.shape, R_CP):,}. "
           "Did you use bias=False and groups=R on the depthwise layers?")
    except Exception as error:  # a learner's function can fail in any way
        notes.append(f"❌ Your function raised {type(error).__name__}: {error}")
    print("\n".join(notes))
    print()
    print("EN: this checks the wiring, not the approximation. The chain reproduces the CP "
          "reconstruction exactly; how far that reconstruction is from the kernel is the "
          "relative error printed above.")
    print("ES: esto comprueba el cableado, no la aproximación.")

### Exercise 3 · Wire the four layers / Ejercicio 3 · Cablea las cuatro capas

Write `build_cp_convs(cp)` that takes TensorLy's CP result `cp = (weights, [U_out, U_in, U_h, U_w])` and returns an `nn.Sequential` of the four `nn.Conv2d` layers above, with their weights copied in. Use `bias=False` throughout.

Conv weights in PyTorch have shape `(out_channels, in_channels / groups, kernel_h, kernel_w)`. So the first layer's weight is `(R, 256, 1, 1)`, a depthwise 3×1 layer's is `(R, 1, 3, 1)`, and so on. The helper `t32(array)` turns a NumPy array into a float32 tensor.

Then run `check_cp_convs(build_cp_convs)`. It feeds a random batch through your chain and through one ordinary 3×3 conv whose kernel is the CP reconstruction, and compares.

<details>
<summary>Español</summary>

Escribe `build_cp_convs(cp)`, que recibe el resultado CP de TensorLy y devuelve un `nn.Sequential` con las cuatro capas `nn.Conv2d` y sus pesos copiados, sin sesgo. Los pesos de conv en PyTorch tienen forma `(salida, entrada / groups, alto, ancho)`. Después ejecuta `check_cp_convs(build_cp_convs)`.

</details>

<details>
<summary>Hint / Pista</summary>

Fold the CP weights into one factor first: `U_out = U_out * weights`. Then the four weights are `U_in.T[:, :, None, None]`, `U_h.T[:, None, :, None]`, `U_w.T[:, None, None, :]` and `U_out[:, :, None, None]`. The depthwise layers need `groups=R` and padding `(1, 0)` and `(0, 1)` so the output keeps its size.

Mete primero los pesos CP en un factor: `U_out = U_out * weights`. Las capas depthwise necesitan `groups=R` y relleno `(1, 0)` y `(0, 1)`.

</details>

In [ ]:
# TODO 3 / TAREA 3
#
# def build_cp_convs(cp):
#     weights, (U_out, U_in, U_h, U_w) = cp
#     ... four nn.Conv2d layers, bias=False, in an nn.Sequential ...
#     ... copy the weights in under torch.no_grad() ...
#     return chain
#
# Then run: check_cp_convs(build_cp_convs)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def build_cp_convs(cp):
    weights, (U_out, U_in, U_h, U_w) = cp
    U_out = U_out * weights                      # fold the CP weights into one factor
    R, (n_out, n_in), (h, w) = U_in.shape[1], (U_out.shape[0], U_in.shape[0]), (U_h.shape[0], U_w.shape[0])
    chain = nn.Sequential(
        nn.Conv2d(n_in, R, 1, bias=False),
        nn.Conv2d(R, R, (h, 1), padding=(h // 2, 0), groups=R, bias=False),
        nn.Conv2d(R, R, (1, w), padding=(0, w // 2), groups=R, bias=False),
        nn.Conv2d(R, n_out, 1, bias=False),
    )
    with torch.no_grad():
        chain[0].weight.copy_(t32(U_in.T[:, :, None, None]))
        chain[1].weight.copy_(t32(U_h.T[:, None, :, None]))
        chain[2].weight.copy_(t32(U_w.T[:, None, None, :]))
        chain[3].weight.copy_(t32(U_out[:, :, None, None]))
    return chain


check_cp_convs(build_cp_convs)

### Tucker-2 is three convolutions / Tucker-2 son tres convoluciones

Tucker-2 compresses only the two **channel** modes and keeps the 3 × 3 spatial modes whole (Kim et al. 2016):

$$
\mathcal{K} \approx \mathcal{G} \times_1 U^{\text{out}} \times_2 U^{\text{in}},
\qquad \mathcal{G} \in \mathbb{R}^{R_o \times R_i \times 3 \times 3} .
$$

The same substitution gives three layers: a **1×1 conv** 256 → $R_i$ (weights $U^{\text{in}}$), an ordinary **3×3 conv** $R_i \to R_o$ whose kernel *is the core*, and a **1×1 conv** $R_o$ → 256 (weights $U^{\text{out}}$). It stores $256 R_i + 256 R_o + 9 R_o R_i$ numbers. Kim et al. leave the spatial modes alone because they "are already quite small": a length-3 mode has little to give.

The fit below is **HOOI** written out in NumPy — alternate between the two channel modes, each time taking the leading left singular vectors of the kernel projected onto the other mode's current subspace — started from HOSVD. Its factors are orthonormal, which will matter in a moment.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Tucker-2 comprime solo los dos modos de <b>canal</b> y deja enteros los espaciales 3 × 3 (Kim et al. 2016). La misma sustitución da tres capas: conv 1×1 de 256 a R_i, una conv 3×3 normal de R_i a R_o cuyo kernel <i>es el núcleo</i>, y conv 1×1 de R_o a 256. Guarda 256R_i + 256R_o + 9R_oR_i números. El ajuste de abajo es <b>HOOI</b> escrito en NumPy, iniciado con HOSVD; sus factores son ortonormales, y eso importará enseguida.</div>

In [ ]:
def tucker2(kernel, r_out, r_in, sweeps=5):
    """HOOI on the two channel modes, from HOSVD. The spatial modes stay whole."""
    U_out = leading(tl.unfold(kernel, 0), r_out)
    U_in = leading(tl.unfold(kernel, 1), r_in)
    for _ in range(sweeps):
        U_out = leading(tl.unfold(np.einsum("tsij,sb->tbij", kernel, U_in), 0), r_out)
        U_in = leading(tl.unfold(np.einsum("tsij,ta->asij", kernel, U_out), 1), r_in)
    core = np.einsum("tsij,ta,sb->abij", kernel, U_out, U_in, optimize=True)
    return core, U_out, U_in


def tucker2_tensor(core, U_out, U_in):
    return np.einsum("abij,ta,sb->tsij", core, U_out, U_in, optimize=True)


core40, V_out, V_in = tucker2(kernel, 40, 43)
tucker_convs = nn.Sequential(
    nn.Conv2d(C_in, 43, 1, bias=False),
    nn.Conv2d(43, 40, 3, padding=1, bias=False),
    nn.Conv2d(40, C_out, 1, bias=False),
)
with torch.no_grad():
    tucker_convs[0].weight.copy_(t32(V_in.T[:, :, None, None]))
    tucker_convs[1].weight.copy_(t32(core40))
    tucker_convs[2].weight.copy_(t32(V_out[:, :, None, None]))
    x = torch.randn(4, C_in, 14, 14, generator=torch.Generator().manual_seed(0))
    gap = (tucker_convs(x) - F.conv2d(x, t32(tucker2_tensor(core40, V_out, V_in)), padding=1)).abs().max()

print(f"Tucker-2 (40, 43): {sum(p.numel() for p in tucker_convs.parameters()):,} weights, "
      f"wiring gap / diferencia de cableado {gap.item():.1e}")
print(f"Relative error / Error relativo: {rel_error(kernel, tucker2_tensor(core40, V_out, V_in)):.3f}")
print(f"‖core‖ / ‖K‖ = {np.linalg.norm(core40) / kernel_norm:.3f}  (orthonormal factors: never above 1)")

### Matched budgets: the fair comparison / Presupuestos iguales: la comparación justa

Exercise 1 showed that at equal rank Tucker-2 costs far more. But equal rank is the wrong comparison, because the two models fit to different errors. The fair question **fixes the number of stored numbers and asks which fits better**.

At 32× compression the budget is CP's own count, 18,648. The next cell scans every Tucker-2 pair $(R_o, R_i)$ that stays within that budget — for each $R_o$, the largest $R_i$ that fits — fits each one, and keeps the best. So Tucker-2 never gets more numbers than CP.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>A igual rango Tucker-2 cuesta mucho más, pero la comparación justa <b>fija el número de números guardados y pregunta cuál ajusta mejor</b>. Con compresión 32× el presupuesto es el de CP, 18.648. La celda siguiente recorre todos los pares (R_o, R_i) de Tucker-2 que caben —para cada R_o, el mayor R_i posible—, ajusta cada uno y se queda con el mejor.</div>

In [ ]:
budget = cp_numbers(kernel.shape, R_CP)
start, tried = time.time(), {}
for r_out in range(8, 65, 4):
    fits = [r for r in range(C_in, 0, -1) if tucker_numbers(kernel.shape, (r_out, r, kh, kw)) <= budget]
    if fits:
        core_, U_o, U_i = tucker2(kernel, r_out, fits[0], sweeps=3)
        tried[(r_out, fits[0])] = rel_error(kernel, tucker2_tensor(core_, U_o, U_i))
best_pair = min(tried, key=tried.get)

print(f"Budget / Presupuesto: {budget:,} numbers ({kernel.size / budget:.0f}× compression)")
print(f"CP, rank {R_CP}:            error {cp32_error:.3f}   largest term {cp32_largest:.2f} × ‖K‖")
print(f"Tucker-2, best {best_pair}: error {tried[best_pair]:.3f}   "
      f"({tucker_numbers(kernel.shape, (*best_pair, kh, kw)):,} numbers; "
      f"{len(tried)} pairs tried in {time.time() - start:.0f} s)")
print("Lower error at 32× / Menor error a 32×:", "CP" if cp32_error < tried[best_pair] else "Tucker-2")

One budget, 100 iterations, one random start: suggestive, not conclusive. The blog post ran the full version, which takes hours on a laptop — six budgets from 64× down to 2×, CP-ALS for **1,000** iterations from **three** random starts each, and every Tucker-2 pair within budget. The next cell downloads those cached results (four small CSVs, hash-checked) and draws them.

The right panel is the number you started recording: CP's **largest rank-one term relative to $\|\mathcal{K}\|$**. A single term many times larger than the whole tensor it helps build can only be there because it **cancels** against another large term. Tucker-2's orthonormal factors make that impossible: its core's norm is at most $\|\mathcal{K}\|$, as the cell above printed.

Cancellation is not a numerical accident; it is built into CP. The smallest example is a rank-3 tensor that is the **limit of rank-2 tensors**:

$$
\mathcal{T} = a\circ a\circ b + a\circ b\circ a + b\circ a\circ a
= \lim_{\varepsilon \to 0} \frac{1}{\varepsilon}\Bigl[(a + \varepsilon b)^{\circ 3} - a^{\circ 3}\Bigr].
$$

Expand the cube and the $1/\varepsilon$ terms cancel, leaving $\mathcal{T}$ plus terms of order $\varepsilon$. So $\mathcal{T}$ can be approximated as well as you like at rank 2 — but only with two terms whose norms grow like $1/\varepsilon$, and **no best rank-2 approximation exists**. ALS chasing a better fit walks down exactly this road; the animation draws it.

![As epsilon shrinks from 1 to 0.02, two rank-one terms grow to about 17 times the target's norm while their sum's error falls to about 3 percent.](https://project-delphi.github.io/tensors-workshop/images/cube-19-cancel.gif)

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un presupuesto, 100 iteraciones y un solo arranque: sugerente, no concluyente. La entrada del blog hizo la versión completa —seis presupuestos de 64× a 2×, CP-ALS con <b>1.000</b> iteraciones y <b>tres</b> arranques, y todos los pares de Tucker-2 dentro del presupuesto—, que tarda horas. La celda siguiente descarga esos resultados y los dibuja.
<br><br>
El panel derecho es el número que empezaste a registrar: el <b>mayor término de rango uno</b> de CP relativo a la norma del kernel. Un término muchas veces mayor que el tensor completo solo puede estar ahí porque se <b>cancela</b> con otro término grande. Los factores ortonormales de Tucker-2 lo impiden.
<br><br>
La cancelación no es un accidente numérico: está en la naturaleza de CP. Un tensor de rango 3 puede ser el <b>límite de tensores de rango 2</b> cuyos dos términos crecen como 1/ε mientras su suma converge: se aproxima tanto como se quiera con rango 2, pero <b>no existe una mejor aproximación de rango 2</b>. ALS, buscando un ajuste mejor, recorre justo ese camino; la animación lo dibuja.</div>

In [ ]:
SWEEP_SHA256 = {
    "sweep_cp_seed0.csv": "f5fb2d3eb3a1ceb09608c12a2c48513dd9f59a411489a7063c2dc8b192e1e832",
    "sweep_cp_seed1.csv": "6adfe63c99a8749f99418a581cf66f1476dbb924f9f0bddc33e99d0acf7e8780",
    "sweep_cp_seed2.csv": "6b942cafbace76f821ff96ab2c130a2af69ea5b4a1866a524bbc14611ceddd89",
    "sweep_tucker2.csv": "1a3fb982baa9ce1e8ff2c0a3ff75aca31b7f59f5a3af97e9bb423a7f2f268cfe",
}
# The workshop's own copies, one per file, tried if GitHub raw does not answer.
SWEEP_COPY = {
    "sweep_cp_seed0.csv": "https://project-delphi.github.io/tensors-workshop/data/sweep_cp_seed0.csv",
    "sweep_cp_seed1.csv": "https://project-delphi.github.io/tensors-workshop/data/sweep_cp_seed1.csv",
    "sweep_cp_seed2.csv": "https://project-delphi.github.io/tensors-workshop/data/sweep_cp_seed2.csv",
    "sweep_tucker2.csv": "https://project-delphi.github.io/tensors-workshop/data/sweep_tucker2.csv",
}
# Four small files in one cell. Two tries of 20 s and one at the copy is 62 s a
# file at worst, 248 s for the four -- inside a runner's five-minute cell limit.
sweeps = {name: pd.read_csv(io.BytesIO(fetch(BLOG_DATA + name, sha, tries=2, timeout=20,
                                             copy=SWEEP_COPY[name])))
          for name, sha in SWEEP_SHA256.items()}
cp_runs = pd.concat(sweeps[f"sweep_cp_seed{s}.csv"] for s in range(3))
t2_runs = sweeps["sweep_tucker2.csv"]

cp_sum = (cp_runs.groupby("ratio")
          .agg(rank=("rank", "first"), numbers=("params", "first"),
               err_lo=("rel_error", "min"), err_hi=("rel_error", "max"),
               term_lo=("max_component_norm", "min"), term_hi=("max_component_norm", "max"))
          .sort_index(ascending=False))
t2_best = (t2_runs.loc[t2_runs.groupby("ratio").rel_error.idxmin()]
           .set_index("ratio").sort_index(ascending=False))
ratios = cp_sum.index.to_numpy()

table = pd.DataFrame({
    "compression": [f"{r}×" for r in ratios],
    "CP rank": cp_sum["rank"].to_numpy(),
    "CP error": [f"{a:.3f}–{b:.3f}" for a, b in zip(cp_sum.err_lo, cp_sum.err_hi)],
    "CP largest term / ‖K‖": [f"{a:.1f}–{b:.1f}" for a, b in zip(cp_sum.term_lo, cp_sum.term_hi)],
    "Tucker-2 (R_o, R_i)": [f"({o}, {i})" for o, i in zip(t2_best.r_out.loc[ratios], t2_best.r_in.loc[ratios])],
    "Tucker-2 error": [f"{e:.3f}" for e in t2_best.rel_error.loc[ratios]],
})
print(table.to_string(index=False))
print()
print("CP fits better at every budget / CP ajusta mejor en todos los presupuestos:",
      bool((t2_best.rel_error.loc[ratios].to_numpy() > cp_sum.err_hi.to_numpy()).all()))

fig, (ax, bx) = plt.subplots(1, 2, figsize=(10, 3.8))
ax.fill_between(ratios, cp_sum.err_lo, cp_sum.err_hi, color=ACCENT, alpha=0.25, lw=0)
ax.plot(ratios, cp_sum.err_lo, "o-", color=ACCENT, label="CP (3 starts)")
ax.plot(ratios, t2_best.rel_error.loc[ratios], "s-", color=CORAL, label="Tucker-2 (best pair)")
ax.scatter([32], [cp32_error], marker="*", s=160, color=ACCENT, edgecolor="k", zorder=5,
           label="your CP fit, 100 iterations")
ax.set_xscale("log", base=2)
ax.set_xticks(ratios, [f"{r}×" for r in ratios])
ax.invert_xaxis()
ax.set_xlabel("compression (dense weights ÷ stored numbers)")
ax.set_ylabel("‖K − K̂‖ / ‖K‖")
ax.set_title("Weight error at matched budgets")
ax.legend(frameon=False)
bx.fill_between(ratios, cp_sum.term_lo, cp_sum.term_hi, color=ACCENT, alpha=0.25, lw=0)
bx.plot(ratios, cp_sum.term_hi, "o-", color=ACCENT)
bx.axhline(1, color=MUTED, ls="--", lw=1)
bx.set_yscale("log")
bx.set_xscale("log", base=2)
bx.set_xticks(ratios, [f"{r}×" for r in ratios])
bx.invert_xaxis()
bx.set_xlabel("compression")
bx.set_ylabel("largest CP term / ‖K‖")
bx.set_title("CP's cancelling terms (dashed: ‖K‖ itself)")
plt.tight_layout()
plt.show()

**CP fits the kernel more closely than Tucker-2 at every budget.** CP at 4× is about as close to the kernel as Tucker-2 at 2×: Tucker-2 needs roughly twice the numbers for the same error. The three CP starts agree to the third decimal, so the gap is not a lucky seed, and CP's error was still falling at 1,000 iterations — ALS with line search never raises the error, so more iterations could only widen the gap. Your 100-iteration fit (the star) already sits on the curve's side of Tucker-2.

**The price is in the right panel.** At every budget, on every start, CP's largest term is at least four times the norm of the kernel it helps approximate — and on one start at 64× it is over two hundred times. CP reaches its low error by adding **large terms that nearly cancel**. That is *degeneracy*, and it is exactly the failure the "what a wrong answer costs" line warned about: a 1% change to one factor of a term 7 times the kernel's norm moves the kernel by 7% of its norm. Fine-tuning takes gradient steps on those factors, which is why Lebedev et al. report that gradients inside the inserted layers "are prone to gradient explosion", and why Phan et al. (2020) add a penalty against it. Your own fit's largest term, printed above, is the same phenomenon caught early.

**And uniqueness runs out.** Kruskal's condition, which guarantees CP components are essentially unique, needs $\sum_n k_n \ge 2R + N - 1$ for an $N$-way array (Sidiropoulos and Bro 2000). With generic factors on this kernel that is $256 + 256 + 3 + 3 \ge 2R + 3$, which fails past $R = 257$. The 4× and 2× fits sit past it: their components carry no uniqueness guarantee at all.

So on this kernel, at fixed error, CP needs about half the numbers — and it gets them by cancellation, not because the kernel is a short sum of rank-one terms.

- **Choose CP** when the compression ratio is the target *and* your pipeline controls degeneracy: a norm penalty, Phan et al.'s stability term, a low fine-tuning learning rate, or frozen inserted layers.
- **Choose Tucker-2** when a well-conditioned starting point for fine-tuning matters more than the last factor of two in storage.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>CP ajusta el kernel mejor que Tucker-2 en todos los presupuestos</b>: CP a 4× se acerca tanto como Tucker-2 a 2×, así que Tucker-2 necesita el doble de números para el mismo error. Los tres arranques coinciden y el error de CP aún bajaba a las 1.000 iteraciones.
<br><br>
<b>El precio está en el panel derecho.</b> En todos los presupuestos y arranques, el mayor término de CP es al menos cuatro veces la norma del kernel (y en un arranque a 64×, más de doscientas): CP consigue su error bajo sumando <b>términos grandes que casi se cancelan</b>. Eso es <i>degeneración</i>: un cambio del 1% en un factor de un término 7 veces mayor que el kernel mueve el kernel un 7%. Por eso el reajuste fino puede explotar (Lebedev et al.) y por eso Phan et al. (2020) añaden una penalización. Además, la condición de Kruskal falla a partir de R = 257, así que los ajustes a 4× y 2× no tienen garantía de unicidad.
<br><br>
<b>Elige CP</b> cuando la razón de compresión es el objetivo y tu proceso controla la degeneración. <b>Elige Tucker-2</b> cuando importa más un punto de partida bien condicionado para el reajuste que el último factor dos de almacenamiento.</div>

## Part 4 · Choosing the rank / Elegir el rango

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

Every comparison so far took the rank as given. In practice it is the hardest choice, and CP and Tucker offer different tools for it.

### The HOSVD scree / El codo de HOSVD

HOSVD gives **one singular-value spectrum per mode**, and a scree plot reads a rank off each:

1. Unfold $\mathcal{K}$ along mode $n$ into an $I_n \times \prod_{m \ne n} I_m$ matrix $\mathcal{K}_{(n)}$.
2. Take its singular values $\sigma_{n,1} \ge \sigma_{n,2} \ge \cdots$.
3. Plot them, and the **retained energy** $E_n(R) = \sum_{i \le R} \sigma_{n,i}^2 \big/ \sum_i \sigma_{n,i}^2$.
4. Pick $R_n$ at the *elbow*, where the spectrum drops to a floor.

What makes this principled is a guarantee (De Lathauwer et al. 2000): truncating every mode's subspace to $R_n$ loses **at most the energy you threw away, summed over modes**,

$$
\frac{\|\mathcal{K} - \hat{\mathcal{K}}\|^2}{\|\mathcal{K}\|^2} \;\le\; \sum_n \bigl(1 - E_n(R_n)\bigr) .
$$

Why? Projecting onto mode $n$'s leading subspace removes exactly $1 - E_n(R_n)$ of the energy of $\mathcal{K}$, the projections onto different modes commute, and each later projection can only remove *less* than it would from the unprojected tensor. Squared errors of orthogonal pieces add, hence the sum.

**CP has no analogue.** There is no CP spectrum: the best rank-$R$ CP approximation need not contain the best rank-$(R-1)$ one, and, as Part 3 showed, it need not even exist in a well-behaved form. That asymmetry is why the second half of this part exists.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>HOSVD da <b>un espectro de valores singulares por modo</b>: despliega el tensor por el modo n, toma sus valores singulares, dibuja la <b>energía retenida</b> E_n(R) y elige R_n en el <i>codo</i>. Lo que lo hace riguroso es una garantía (De Lathauwer et al. 2000): truncar cada modo pierde <b>como mucho la energía descartada, sumada sobre los modos</b>. Proyectar sobre el subespacio principal del modo n quita exactamente 1 − E_n(R_n) de la energía, las proyecciones de modos distintos conmutan y cada una quita menos del tensor ya proyectado; los errores al cuadrado de piezas ortogonales se suman.
<br><br>
<b>CP no tiene análogo</b>: no hay espectro CP, la mejor aproximación de rango R no contiene necesariamente la de rango R − 1 y, como vio la Parte 3, puede ni existir de forma estable.</div>

In [ ]:
spectra = {n: np.linalg.svd(tl.unfold(kernel, n), compute_uv=False) for n in range(4)}
energy = {n: np.cumsum(s ** 2) / np.sum(s ** 2) for n, s in spectra.items()}


def rank_at(n, share):
    """Smallest R_n whose retained energy reaches `share`."""
    return int(np.searchsorted(energy[n], share)) + 1


for share in (0.50, 0.90, 0.95):
    print(f"{share:.0%} energy / energía: (R_out, R_in) = ({rank_at(0, share)}, {rank_at(1, share)})")
print(f"Spatial modes need {rank_at(2, 0.9)} and {rank_at(3, 0.9)} of their 3 slices for 90%")

# Truncate the two channel modes at their 90% ranks, and test the bound.
r90 = (rank_at(0, 0.90), rank_at(1, 0.90))
P_out, P_in = (leading(tl.unfold(kernel, n), r) for n, r in zip((0, 1), r90))
K90 = np.einsum("tT,sS,TSij->tsij", P_out @ P_out.T, P_in @ P_in.T, kernel, optimize=True)
error_90 = rel_error(kernel, K90)
bound_90 = np.sqrt(sum(1 - energy[n][r - 1] for n, r in zip((0, 1), r90)))
print(f"At the 90% ranks: Tucker-2 stores {tucker_numbers(kernel.shape, (*r90, 3, 3)):,} numbers "
      f"({kernel.size / tucker_numbers(kernel.shape, (*r90, 3, 3)):.2f}× compression)")
print(f"Truncation error {error_90:.3f} ≤ bound {bound_90:.3f} / Error de truncamiento ≤ cota:",
      error_90 <= bound_90)

fig, (ax, bx) = plt.subplots(1, 2, figsize=(10, 3.6))
index = np.arange(1, C_out + 1)
for n, name, colour in [(0, "out-channels", ACCENT), (1, "in-channels", TEAL)]:
    ax.plot(index, spectra[n] / spectra[n][0], color=colour, label=name)
    bx.plot(index, energy[n], color=colour, label=name)
ax.set_yscale("log")
ax.set_xlabel("index i")
ax.set_ylabel("σ_i / σ_1")
ax.set_title("No elbow")
ax.legend(frameon=False)
for share in (0.90, 0.95):
    bx.axhline(share, color=MUTED, ls="--", lw=1)
bx.set_xlabel("rank R_n")
bx.set_ylabel("retained energy E_n(R_n)")
bx.set_title("Retained energy")
plt.tight_layout()
plt.show()

**The kernel's spectra have no elbow.** Past the first few values both channel spectra fall smoothly all the way to index 256; no index separates structure from a noise floor. 90% of the energy already needs more than 150 components per channel mode, and at those ranks Tucker-2 barely compresses at all. The bound holds, as it must — but it holds at ranks you would not choose.

A scree finds a rank **when the tensor has one**: a low-rank signal above a noise floor. A trained kernel spreads its energy over most of its channels, so the scree turns into an *energy budget*, and the budget is a guess about accuracy after fine-tuning. Kim et al. choose Tucker-2 ranks with a noise model instead (variational Bayesian matrix factorisation on the two unfoldings).

On the knowledge graph a scree would say nothing at all: its mode-1 unfolding is 14,505 × 3.4 million and its zeros are *missing* entries, not measured ones. Ranks there come from validation MRR.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Los espectros del kernel no tienen codo</b>: caen suavemente hasta el índice 256, y el 90% de la energía ya pide más de 150 componentes por modo de canal, rangos a los que Tucker-2 apenas comprime. La cota se cumple, pero en rangos que no elegirías. Un codo encuentra un rango <b>cuando el tensor lo tiene</b>: una señal de rango bajo sobre un suelo de ruido. En el grafo de conocimiento no diría nada, porque sus ceros son entradas <i>faltantes</i>; allí el rango sale del MRR de validación.</div>

### Over-parameterise, then prune / Sobreparametriza y poda

Grid search does not scale: Tucker-2 on this kernel alone has $256^2 = 65{,}536$ rank pairs, and one CP-ALS fit at the 4× rank takes nine minutes. The alternative is **one fit with too many components and a penalty that switches whole components off**. For CP there is a clean argument for why this works.

**Scale freedom.** Multiply the mode-$n$ column of term $r$ by $c_n > 0$ with $\prod_n c_n = 1$ and the fit does not change (Part 1, line 4). So any penalty on the factors is, in effect, minimised over that freedom — and the AM–GM inequality says what it becomes. For the column norms of term $r$,

$$
\sum_{n=1}^{N} \bigl\|a^{(n)}_r\bigr\|_2 \;\ge\; N \Bigl(\prod_n \bigl\|a^{(n)}_r\bigr\|_2\Bigr)^{1/N} = N \lambda_r^{1/N},
$$

with equality when the $N$ norms are equal. Here $\lambda_r$ is the Frobenius norm of term $r$'s rank-one tensor. So:

- **Group lasso** — the $\ell_2$ norm of every column — charges term $r$ exactly $N\lambda_r^{1/N}$ at a balanced point. At $N = 3$ that is a **cube root**: its slope is infinite at zero, so small terms are pushed all the way to zero rather than merely shrunk.
- **Weight decay** — squared $\ell_2$ — gives $N\lambda_r^{2/N}$. For a matrix ($N = 2$) that is $2\lambda_r$, the nuclear norm. From $N = 3$ on the exponent is below one, so **plain weight decay prunes CP terms too**.
- **Elementwise $\ell_1$** gives the same bound in $\ell_1$ norms, and also zeroes entries inside the terms that survive.

Every local minimum is balanced (otherwise rescaling would lower the penalty for free), so the bound is attained there. An optimiser's iterates need not be balanced, so the fits below **rebalance** after every step: rescale each term's columns to equal norm, which leaves the fit unchanged and can only lower the penalty.

Tucker has no such reduction — its freedom is an invertible matrix per mode, not a scale per term — which is why this tool belongs to CP.

**Proximal steps.** A gradient step never lands exactly on zero. **Proximal gradient descent** takes a gradient step on the smooth fit, then applies the penalty's *proximal map*, which can. For group lasso with step $\eta$ and strength $\mu$ it acts on each column $a$ at once:

$$
a \;\leftarrow\; a \cdot \max\Bigl(0,\; 1 - \frac{\eta\mu}{\|a\|_2}\Bigr).
$$

A column shorter than $\eta\mu$ becomes exactly zero; a longer one keeps its direction and loses $\eta\mu$ of its length. Weight decay has no zeroing map; its terms decay towards zero, and a term counts as gone once $\lambda_r < 10^{-6}$.

**The test tensor is synthetic, on purpose**: grading a rank needs the true one. Six rank-one terms with Gaussian factors on a 60 × 50 × 40 grid, scaled to unit norm, plus Gaussian noise of norm 0.1 (20 dB SNR). The fit starts at rank **30**, five times too many.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La búsqueda en rejilla no escala. La alternativa es <b>un solo ajuste con demasiadas componentes y una penalización que apaga componentes enteras</b>. Para CP hay un argumento limpio: por la libertad de escala, cualquier penalización sobre los factores se minimiza sobre esa libertad, y la desigualdad AM–GM dice en qué se convierte. <b>Group lasso</b> (norma ℓ2 de cada columna) cobra N·λ_r^(1/N) al término r: con N = 3 es una raíz cúbica, de pendiente infinita en cero, así que empuja los términos pequeños hasta cero. <b>Weight decay</b> cobra N·λ_r^(2/N): para matrices es la norma nuclear, y desde N = 3 el exponente es menor que uno, así que también poda. Los ajustes <b>reequilibran</b> tras cada paso. Tucker no tiene esta reducción.
<br><br>
<b>Pasos proximales</b>: un paso de gradiente nunca cae en cero; el mapa proximal de group lasso sí: a ← a · max(0, 1 − ημ/‖a‖). Una columna más corta que ημ se anula; una más larga conserva su dirección y pierde ημ de longitud.
<br><br>
El tensor de prueba es sintético <b>a propósito</b>, porque evaluar un rango exige conocer el verdadero: seis términos de rango uno en 60 × 50 × 40, norma 1, más ruido de norma 0,1. El ajuste empieza con rango <b>30</b>.</div>

In [ ]:
SHAPE, TRUE_RANK, FIT_RANK, NOISE, ALIVE = (60, 50, 40), 6, 30, 0.1, 1e-6
f64 = torch.float64


def make_problem(seed):
    """Noiseless tensor, noisy tensor, and a small random rank-30 start."""
    g = torch.Generator().manual_seed(seed)
    truth = [torch.randn(n, TRUE_RANK, generator=g, dtype=f64) for n in SHAPE]
    clean = cp3(truth)
    clean = clean / clean.norm()
    noise = torch.randn(SHAPE, generator=g, dtype=f64)
    noisy = clean + NOISE * noise / noise.norm()
    start = [0.3 * torch.randn(n, FIT_RANK, generator=g, dtype=f64) for n in SHAPE]
    return clean, noisy, start


def cp3(factors):
    return torch.einsum("ir,jr,kr->ijk", *factors)


def term_weights(factors):
    """λ_r: the product over modes of column r's ℓ2 norm."""
    return torch.stack([f.norm(dim=0) for f in factors]).prod(0)


clean, noisy, start = make_problem(0)
print(f"Noisy tensor / Tensor ruidoso: {tuple(noisy.shape)}, true rank {TRUE_RANK}, "
      f"fit rank {FIT_RANK}, ‖noise‖ = {(noisy - clean).norm().item():.2f}")

In [ ]:
# Checker for Exercise 4 / Comprobador del Ejercicio 4.


def check_prox(prox_group):
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    a = torch.tensor([[3.0, 0.3], [4.0, 0.4]], dtype=f64)      # column norms 5 and 0.5
    try:
        out = prox_group([a], step=0.1, strength=10.0)          # threshold ημ = 1
        ok(isinstance(out, list) and len(out) == 1,
           "prox_group returns a list with one matrix per mode.",
           "prox_group should return a list of factor matrices, one per mode.")
        col0, col1 = out[0][:, 0], out[0][:, 1]
        ok(torch.allclose(col1, torch.zeros(2, dtype=f64)),
           "The column shorter than ημ = 1 is exactly zero.",
           f"The column of norm 0.5 became {col1.tolist()}; with ημ = 1 it should be exactly zero.")
        ok(torch.allclose(col0, torch.tensor([2.4, 3.2], dtype=f64)),
           "The column of norm 5 keeps its direction and shrinks to norm 4.",
           f"The column of norm 5 became {col0.tolist()}; it should be (3, 4) · (1 − 1/5) = (2.4, 3.2).")
        ok(torch.equal(a, torch.tensor([[3.0, 0.3], [4.0, 0.4]], dtype=f64)),
           "The input was not modified in place.",
           "The input matrix changed: build new tensors rather than editing in place.")
    except Exception as error:  # a learner's function can fail in any way
        notes.append(f"❌ Your function raised {type(error).__name__}: {error}")
    print("\n".join(notes))

### Exercise 4 · The group-lasso proximal map / Ejercicio 4 · El mapa proximal de group lasso

Write `prox_group(factors, step, strength)`: for each factor matrix in the list, scale every **column** $a$ by $\max\bigl(0,\, 1 - \text{step} \cdot \text{strength} / \|a\|_2\bigr)$, and return the new list. Guard against a zero column (its norm is 0): it should stay zero, not become NaN.

**Predict first:** a column of norm 0.5 and one of norm 5, with $\eta\mu = 1$. What does each become?

Then run `check_prox(prox_group)`. The fit in the next cell calls your function by that name.

<details>
<summary>Español</summary>

Escribe `prox_group(factors, step, strength)`: para cada matriz de la lista, escala cada <b>columna</b> a por max(0, 1 − step·strength / ‖a‖) y devuelve la lista nueva, sin producir NaN en una columna nula. <b>Predice</b>: con ημ = 1, ¿en qué se convierten una columna de norma 0,5 y otra de norma 5? Después ejecuta `check_prox(prox_group)`.

</details>

<details>
<summary>Hint / Pista</summary>

`f.norm(dim=0)` gives every column's norm at once; `.clamp_min(1e-300)` keeps the division finite; `.clamp(min=0)` is the max with zero. Broadcasting a length-$R$ vector against an $I \times R$ matrix scales columns.

`f.norm(dim=0)` da la norma de todas las columnas; `.clamp_min(1e-300)` evita dividir por cero; `.clamp(min=0)` es el máximo con cero.

</details>

In [ ]:
# TODO 4 / TAREA 4
#
# def prox_group(factors, step, strength):
#     return [ ... each factor with every column scaled by
#              max(0, 1 - step * strength / ‖column‖) ... ]
#
# Then run: check_prox(prox_group)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def prox_group(factors, step, strength):
    return [f * (1 - step * strength / f.norm(dim=0).clamp_min(1e-300)).clamp(min=0)
            for f in factors]


check_prox(prox_group)

In [ ]:
# Visible on purpose: the fits below call prox_group.
# Visible a propósito: los ajustes de abajo llaman a prox_group.


def prox_group(factors, step, strength):
    """Group lasso's proximal map: shrink every column by step·strength, to zero if shorter."""
    return [f * (1 - step * strength / f.norm(dim=0).clamp_min(1e-300)).clamp(min=0)
            for f in factors]


def rebalance(factors):
    """Give each term's columns equal norm; the product, and so the fit, is unchanged.

    A term with one all-zero column is zero already, so all its columns are zeroed.
    """
    norms = torch.stack([f.norm(dim=0) for f in factors])
    dead = (norms == 0).any(0)
    norms = norms.clamp_min(1e-300)
    target = norms.log().mean(0).exp().masked_fill(dead, 0.0)
    return [f * (target / n) for f, n in zip(factors, norms)]


def fit_pruned(data, start, kind, strength, iters=1500):
    """Proximal gradient with backtracking on ½‖data − CP‖², then rebalance.

    kind: "none", "group" (group lasso, by prox) or "wd" (weight decay, in the gradient).
    """
    factors, step = [f.clone() for f in start], 1.0

    def smooth(fs):
        value = 0.5 * ((data - cp3(fs)) ** 2).sum()
        if kind == "wd":
            value = value + strength * sum((f ** 2).sum() for f in fs)
        return value

    for _ in range(iters):
        if not any(f.any() for f in factors):
            break                                   # every term is zero: nothing left to move
        live = [f.clone().requires_grad_(True) for f in factors]
        value = smooth(live)
        grads = torch.autograd.grad(value, live)
        value = value.detach()
        while True:                                 # backtrack until the quadratic bound holds
            trial = [f - step * g for f, g in zip(factors, grads)]
            if kind == "group":
                trial = prox_group(trial, step, strength)
            moves = [t - f for t, f in zip(trial, factors)]
            bound = (value + sum((g * m).sum() for g, m in zip(grads, moves))
                     + sum((m ** 2).sum() for m in moves) / (2 * step))
            if smooth(trial) <= bound + 1e-15:
                break
            step *= 0.5
            if not torch.isfinite(bound) or step < 1e-12:
                raise FloatingPointError("EN: the fit diverged. / ES: el ajuste divergió.")
        factors = rebalance(trial)
        step = min(step * 1.5, 1e3)
    return factors

In [ ]:
start_time = time.time()
print(f"{'penalty':<24}{'terms left':>11}{'largest dropped λ':>19}{'distance to truth':>19}")
pruned = {}
for label, kind, strength in [("none", "none", 0.0),
                              ("group lasso, μ = 3e-4", "group", 3e-4),
                              ("weight decay, μ = 1e-3", "wd", 1e-3)]:
    factors = fit_pruned(noisy, start, kind, strength)
    lam = torch.sort(term_weights(factors), descending=True).values.numpy()
    alive = int((lam > ALIVE).sum())
    dropped = f"{lam[alive]:.0e}" if alive < FIT_RANK else "—"
    pruned[kind] = lam
    print(f"{label:<24}{alive:>5} of {FIT_RANK}{dropped:>19}"
          f"{(clean - cp3(factors)).norm().item():>19.4f}")
print(f"({time.time() - start_time:.0f} s)")

fig, ax = plt.subplots(figsize=(7.5, 3.0))
for kind, colour, name in [("none", MUTED, "no penalty"), ("group", ACCENT, "group lasso"),
                           ("wd", CORAL, "weight decay")]:
    ax.semilogy(np.arange(1, FIT_RANK + 1), np.maximum(pruned[kind], 1e-12), "o-",
                ms=3, color=colour, label=name)
ax.axvline(TRUE_RANK + 0.5, color="k", lw=0.8, ls=":")
ax.set_xlabel("term, sorted by weight λ_r")
ax.set_ylabel("λ_r (floored at 1e-12)")
ax.set_title("30 terms fitted, 6 true: what each penalty keeps")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

One strength found the rank. How sensitive is that to the strength you picked? The next cell sweeps group lasso across three orders of magnitude on the same tensor (about half a minute).

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una fuerza encontró el rango. ¿Cuánto depende de la fuerza elegida? La celda siguiente recorre group lasso a lo largo de tres órdenes de magnitud sobre el mismo tensor.</div>

In [ ]:
strengths = [1e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
rows = []
for strength in strengths:
    factors = fit_pruned(noisy, start, "group", strength, iters=1000)
    rows.append((strength, int((term_weights(factors) > ALIVE).sum()),
                 (clean - cp3(factors)).norm().item()))
sweep = pd.DataFrame(rows, columns=["strength μ", "terms left", "distance to truth"])
print(sweep.to_string(index=False, formatters={"strength μ": "{:g}".format,
                                                "distance to truth": "{:.4f}".format}))

Read the two tables together.

- **Without a penalty all 30 terms survive**, and the fit sits further from the noiseless tensor than the pruned fits: the 24 spare terms are fitting noise.
- **Group lasso finds the rank**, and the dropped terms are *exactly* zero — the proximal map did that. It also **denoises**: the pruned fit is closer to the truth than the unpenalised one.
- **Weight decay finds it too**, with no zeroing map at all: its dropped terms are tiny rather than zero, which is the $\lambda_r^{2/3}$ bound at work for three modes.
- **Too weak keeps spares; too strong cuts true terms.** There is a window of strengths that returns exactly six, and inside it the penalty still shrinks the survivors, so the distance to the truth creeps up towards the top of the window. The usual last step — not done here — is to refit at the found rank with no penalty.

The window moves with the scale of the tensor and with the noise, so on real data it is found against **held-out entries**. That is still a win: one sweep over a single strength instead of a grid over ranks. The blog post runs the full version — three penalties, seven strengths, five seeds — and all three penalties find the rank on every seed.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Sin penalización sobreviven los 30 términos</b> y el ajuste queda más lejos del tensor limpio: los 24 sobrantes ajustan ruido. <b>Group lasso encuentra el rango</b> con los términos descartados <i>exactamente</i> en cero, y además <b>quita ruido</b>. <b>Weight decay también lo encuentra</b>, sin mapa que anule: sus términos descartados son diminutos, no cero. <b>Demasiado débil deja sobrantes; demasiado fuerte corta términos verdaderos</b>: hay una ventana que devuelve exactamente seis, y dentro de ella la penalización aún encoge a los supervivientes. En datos reales la ventana se busca con <b>entradas reservadas</b>: un barrido sobre una sola fuerza en lugar de una rejilla de rangos.</div>

## Part 5 · One photograph, both jobs / Una fotografía, los dos trabajos

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

The knowledge graph only asked for prediction; the kernel only asked for compression. A photograph can be asked **either**, so it separates the two choices on one tensor: same fold, same models, and possibly two different answers about which is better.

**The fold.** Cut the image into **8 × 8 blocks** — the block size JPEG uses — and index a pixel by *block row, block column, position within the block, colour*. The photograph becomes a 4-way tensor:

$$
\text{image}[8I + u,\; 8J + v,\; c] \;=\; \mathcal{X}[I,\, J,\, 8u + v,\, c] .
$$

![A 32 by 32 crop of the astronaut cut into 8 by 8 blocks; one block at a time is outlined, lit in a 4 by 4 grid of block indices, and shown as its column of 64 pixel positions.](https://project-delphi.github.io/tensors-workshop/images/cube-19-fold.gif)

Why these four modes? Block rows and block columns carry the **layout of this particular picture**. The within-block mode carries **texture every block shares** — edges, gradients, flat patches — and the colour mode carries three correlated channels. The last two describe photographs in general.

**The data.** scikit-image's `astronaut`, NASA's public-domain portrait of Eileen Collins, averaged over 2 × 2 pixel squares to 256 × 256 so every fit below takes seconds, not minutes. It folds to $32 \times 32 \times 64 \times 3$ = 196,608 entries. (The blog post uses the full 512 × 512 image; the conclusions are the same.)

Exercise 5 is the fold itself — a reshape-and-transpose, the kind of axis bookkeeping section 04 was about.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>El grafo solo pedía predecir y el kernel solo comprimir. A una fotografía se le puede pedir <b>cualquiera de los dos</b>, así que separa las dos decisiones sobre un mismo tensor.
<br><br>
<b>El plegado</b>: corta la imagen en bloques de <b>8 × 8</b>, como JPEG, e indexa cada píxel por <i>fila de bloque, columna de bloque, posición dentro del bloque y color</i>. Las filas y columnas de bloque llevan la <b>disposición de esta foto concreta</b>; el modo dentro del bloque lleva <b>texturas que comparten todos los bloques</b>, y el color tres canales correlacionados. Los dos últimos describen fotografías en general.
<br><br>
Los datos: <code>astronaut</code> de scikit-image (retrato de dominio público de Eileen Collins), promediado a 256 × 256 para que cada ajuste tarde segundos. Se pliega en 32 × 32 × 64 × 3. El Ejercicio 5 es el plegado mismo: un reshape con transpose, como en la sección 04.</div>

In [ ]:
# Checker for Exercise 5 / Comprobador del Ejercicio 5.


def check_fold(fold, unfold_image):
    notes = []

    def ok(condition, good, bad):
        notes.append(("✅ " + good) if condition else ("❌ " + bad))

    img = np.arange(24 * 16 * 3, dtype=float).reshape(24, 16, 3)   # 3 × 2 blocks of 8 × 8
    try:
        x = fold(img)
        ok(np.shape(x) == (3, 2, 64, 3),
           "A 24 × 16 × 3 image folds to (3, 2, 64, 3).",
           f"fold gave shape {np.shape(x)}; expected (block rows, block cols, 64, colours) = (3, 2, 64, 3).")
        if np.shape(x) == (3, 2, 64, 3):
            I, J, u, v, c = 2, 1, 3, 5, 2
            ok(x[I, J, 8 * u + v, c] == img[8 * I + u, 8 * J + v, c],
               "Entry [2, 1, 8·3 + 5, 2] is pixel (8·2 + 3, 8·1 + 5) in colour 2, as the formula says.",
               f"fold(img)[2, 1, 29, 2] = {x[I, J, 8 * u + v, c]}, but that position is pixel "
               f"(19, 13) in colour 2 = {img[19, 13, 2]}. A plain reshape mixes blocks: you need a transpose.")
            back = unfold_image(x)
            ok(np.array_equal(back, img),
               "unfold_image(fold(img)) gives the image back exactly.",
               "unfold_image(fold(img)) is not the image; undo the transpose before the last reshape.")
    except Exception as error:  # a learner's function can fail in any way
        notes.append(f"❌ Your function raised {type(error).__name__}: {error}")
    print("\n".join(notes))

### Exercise 5 · Fold a photograph into blocks / Ejercicio 5 · Pliega una fotografía en bloques

Write `fold(img, block=8)`, taking an `H × W × C` image with `H` and `W` multiples of `block` to an `(H/block) × (W/block) × block² × C` tensor with the index convention above, and its inverse `unfold_image(x, block=8)`.

**Predict first:** would `img.reshape(H // 8, W // 8, 64, C)` be right? Try it on paper for a 16 × 16 image: which pixels would land in "block (0, 0)"?

Then run `check_fold(fold, unfold_image)`.

<details>
<summary>Español</summary>

Escribe `fold(img, block=8)`, que lleva una imagen `H × W × C` a un tensor `(H/8) × (W/8) × 64 × C` con la convención de arriba, y su inversa `unfold_image(x, block=8)`. <b>Predice</b>: ¿valdría `img.reshape(H // 8, W // 8, 64, C)`? Pruébalo en papel con una imagen 16 × 16. Después ejecuta `check_fold(fold, unfold_image)`.

</details>

<details>
<summary>Hint / Pista</summary>

First expose all five indices with a reshape to `(H/8, 8, W/8, 8, C)` — that is (I, u, J, v, c) — then `transpose(0, 2, 1, 3, 4)` to put them in the order (I, J, u, v, c), then merge u and v with a final reshape. The inverse runs the same three steps backwards.

Primero expón los cinco índices con un reshape a `(H/8, 8, W/8, 8, C)`, es decir (I, u, J, v, c); después `transpose(0, 2, 1, 3, 4)` para el orden (I, J, u, v, c), y un último reshape que junte u y v.

</details>

In [ ]:
# TODO 5 / TAREA 5
#
# def fold(img, block=8):          H × W × C  ->  (H/8) × (W/8) × 64 × C
# def unfold_image(x, block=8):    the inverse
#
# Then run: check_fold(fold, unfold_image)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo tú primero { display-mode: 'form' }


def fold(img, block=8):
    h, w, c = img.shape
    blocks = img.reshape(h // block, block, w // block, block, c)      # I, u, J, v, c
    return blocks.transpose(0, 2, 1, 3, 4).reshape(h // block, w // block, block * block, c)


def unfold_image(x, block=8):
    bh, bw, _, c = x.shape
    blocks = x.reshape(bh, bw, block, block, c).transpose(0, 2, 1, 3, 4)
    return blocks.reshape(bh * block, bw * block, c)


check_fold(fold, unfold_image)

In [ ]:
# Visible on purpose: the rest of Part 5 folds and unfolds with these.
# Visible a propósito: el resto de la Parte 5 pliega y despliega con estas.


def fold(img, block=8):
    """H × W × C image -> (H/block) × (W/block) × block² × C tensor."""
    h, w, c = img.shape
    blocks = img.reshape(h // block, block, w // block, block, c)
    return blocks.transpose(0, 2, 1, 3, 4).reshape(h // block, w // block, block * block, c)


def unfold_image(x, block=8):
    """The inverse of fold."""
    bh, bw, _, c = x.shape
    blocks = x.reshape(bh, bw, block, block, c).transpose(0, 2, 1, 3, 4)
    return blocks.reshape(bh * block, bw * block, c)


photo = skdata.astronaut().astype(np.float64) / 255                 # 512 × 512 × 3
photo = photo.reshape(256, 2, 256, 2, 3).mean(axis=(1, 3))          # 2 × 2 average: 256 × 256 × 3
X_img = fold(photo)

img_energy = {n: np.cumsum(s ** 2) / np.sum(s ** 2)
              for n, s in ((n, np.linalg.svd(tl.unfold(X_img, n), compute_uv=False)) for n in range(4))}
print(f"Folded photograph / Fotografía plegada: {X_img.shape}, {X_img.size:,} entries")
print("Components for 99.9% of each mode's energy / Componentes para el 99,9% de la energía:",
      [int(np.searchsorted(img_energy[n], 0.999)) + 1 for n in range(4)], "of", list(X_img.shape))

The spectra already hint at the answer: keeping 99.9% of the energy needs nearly all 32 components of each **layout** mode, while the **within-block** mode needs far fewer than its 64. Only the texture mode compresses easily.

### Job 1 · Compression / Trabajo 1 · Compresión

Every pixel is fitted, and the question is the **smallest model that is good enough**. "Good enough" is measured by PSNR, the image-processing scale for reconstruction error, in decibels:

$$
\mathrm{PSNR} = 10 \log_{10} \frac{1}{\mathrm{MSE}} \quad\text{for pixel values in } [0, 1];
$$

each +10 dB is a tenfold smaller mean squared error, and around 28–30 dB a photograph starts to look clean. The threshold here is **28 dB**. It is a choice, and the plot shows what a different one would cost.

- **CP** at ranks 50, 100, 150 and 200, by ALS with line search, storing $R(32 + 32 + 64 + 3) = 131R$ numbers.
- **Tucker** on a grid of ranks $(R, R, R_p, R_c)$, with $R$ shared by block rows and columns, fitted by HOSVD plus one HOOI sweep; storage by your `tucker_numbers`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Los espectros ya lo anuncian: conservar el 99,9% de la energía pide casi las 32 componentes de cada modo de <b>disposición</b>, mientras que el modo <b>dentro del bloque</b> pide muchas menos de 64.
<br><br>
<b>Compresión</b>: se ajustan todos los píxeles y la pregunta es el <b>modelo más pequeño que basta</b>, medido con PSNR (10·log10(1/MSE) para píxeles en [0, 1]; cada +10 dB es un error cuadrático diez veces menor). El umbral es <b>28 dB</b>. CP con rangos 50–200 (131R números) y Tucker en una rejilla (R, R, R_p, R_c).</div>

In [ ]:
def psnr(truth, estimate, where=None):
    """Peak signal-to-noise ratio in dB, for images scaled to [0, 1]."""
    err = (np.clip(estimate, 0, 1) - truth) ** 2
    return float(10 * np.log10(1 / (err if where is None else err[where]).mean()))


def fit_tucker(x, ranks, mask=None, sweeps=1):
    """HOSVD, then HOOI sweeps. With a mask, missing entries are re-imputed from the fit (EM)."""
    filled = x.copy() if mask is None else np.where(mask, x, x[mask].mean())
    factors = [leading(tl.unfold(filled, m), r) for m, r in enumerate(ranks)]
    for _ in range(sweeps):
        for m in range(x.ndim):
            projected = tl.tenalg.multi_mode_dot(filled, factors, skip=m, transpose=True)
            factors[m] = leading(tl.unfold(projected, m), ranks[m])
        core = tl.tenalg.multi_mode_dot(filled, factors, transpose=True)
        estimate = tl.tenalg.multi_mode_dot(core, factors)
        if mask is not None:
            filled = np.where(mask, x, estimate)
    return estimate


PSNR_TARGET = 28.0
start_time = time.time()
compress, rebuilt = [], {}
for rank in (50, 100, 150, 200):
    cp = parafac(X_img, rank, init="random", random_state=0, n_iter_max=80, tol=1e-9, linesearch=True)
    rebuilt[("CP", rank)] = tl.cp_to_tensor(cp)
    compress.append(("CP", rank, cp_numbers(X_img.shape, rank), psnr(X_img, rebuilt[("CP", rank)])))
for r in (16, 24, 32):
    for r_p in (4, 8, 12, 16, 24):
        for r_c in (1, 2, 3):
            ranks = (r, r, r_p, r_c)
            rebuilt[("Tucker", ranks)] = fit_tucker(X_img, ranks)
            compress.append(("Tucker", ranks, tucker_numbers(X_img.shape, ranks),
                             psnr(X_img, rebuilt[("Tucker", ranks)])))
compress = pd.DataFrame(compress, columns=["model", "ranks", "numbers", "psnr"])
compress = compress[compress.numbers < X_img.size]

picks = {m: g[g.psnr >= PSNR_TARGET].sort_values("numbers").iloc[0] for m, g in compress.groupby("model")}
for m, row in picks.items():
    print(f"{m:<7} clears {PSNR_TARGET:.0f} dB at {row.ranks}: {row.numbers:,} numbers, "
          f"{X_img.size / row.numbers:.1f}× compression, {row.psnr:.1f} dB")
print(f"Tucker needs {picks['Tucker'].numbers / picks['CP'].numbers:.1f}× CP's storage "
      f"/ Tucker necesita {picks['Tucker'].numbers / picks['CP'].numbers:.1f}× el almacenamiento de CP "
      f"({time.time() - start_time:.0f} s)")
print("Fewer numbers at the threshold / Menos números en el umbral:",
      min(picks, key=lambda m: picks[m].numbers))

tk = compress[compress.model == "Tucker"].sort_values("numbers")
frontier = tk[tk.psnr == tk.psnr.cummax()]
cp_curve = compress[compress.model == "CP"]
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.scatter(tk.numbers, tk.psnr, s=10, color="#c9ccd3", label="Tucker rank tuples")
ax.plot(frontier.numbers, frontier.psnr, "-", color=CORAL, label="Tucker frontier")
ax.plot(cp_curve.numbers, cp_curve.psnr, "o-", color=ACCENT, label="CP")
ax.axhline(PSNR_TARGET, color=MUTED, ls="--", lw=1)
for m, colour in (("CP", ACCENT), ("Tucker", CORAL)):
    ax.scatter(picks[m].numbers, picks[m].psnr, s=160, facecolors="none", edgecolors=colour, lw=1.6)
ax.set_xscale("log")
ax.set_xlabel(f"stored numbers (dense tensor: {X_img.size:,})")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Compression: the smallest model that clears the dashed line")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

**For compression, CP wins again**, and by about the same factor as on the kernel: at the threshold, Tucker needs roughly twice CP's storage. The spectra predicted it. The layout modes need nearly all their length, so a Tucker core over them is close to *the full size of those modes multiplied together* ($32 \times 32 \times R_p \times R_c$). CP pays for each component once per mode instead of once per *combination*.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Para comprimir, CP vuelve a ganar</b>, por un factor parecido al del kernel: en el umbral Tucker necesita más o menos el doble. Los espectros lo anunciaban: los modos de disposición necesitan casi toda su longitud, así que un núcleo de Tucker sobre ellos es casi el producto completo de esos modos, mientras que CP paga cada componente una vez por modo y no una vez por <i>combinación</i>.</div>

### Job 2 · Prediction / Trabajo 2 · Predicción

Now **hide half the pixels**, and ask which model best predicts them.

- **Split.** Every pixel is drawn into *training* (50%), *validation* (25%) or *test* (25%), all three colours together. Models see only training pixels; ranks are chosen on validation; the chosen model is scored once on test.
- **Fitting with holes: expectation–maximisation.** Fill the missing entries with the current estimate, refit as if the tensor were complete, re-estimate, repeat. For CP, TensorLy's `parafac(..., mask=...)` does this inside ALS; for Tucker it is the `mask` branch of `fit_tucker` above, around HOOI.
- **Baseline.** Each hidden pixel set to the mean of its observed 3 × 3 neighbours. Any factorisation has to be compared with *something* this simple.

This is the knowledge-graph job again — score entries nobody observed — on a tensor where you can look at the answer.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Ahora se <b>ocultan la mitad de los píxeles</b>. Cada píxel va a <i>entrenamiento</i> (50%), <i>validación</i> (25%) o <i>prueba</i> (25%); los rangos se eligen en validación y el modelo elegido se puntúa una vez en prueba. Para ajustar con huecos se usa <b>esperanza–maximización</b>: rellena lo que falta con la estimación actual, reajusta como si el tensor estuviera completo y repite. La referencia es la media de los vecinos 3 × 3 observados. Es el trabajo del grafo de conocimiento otra vez, sobre un tensor donde se puede mirar la respuesta.</div>

In [ ]:
labels = np.random.default_rng(0).choice(3, size=photo.shape[:2], p=(0.5, 0.25, 0.25))


def entries(which):
    """Pixel labels -> a boolean mask over the folded tensor (all colours together)."""
    return fold(np.repeat((labels == which)[..., None], 3, axis=2).astype(float)) > 0.5


train, val, test = entries(0), entries(1), entries(2)


def neighbour_mean(img, observed):
    """Baseline: each missing pixel is the mean of its observed 3 × 3 neighbours."""
    known, estimate = observed.copy(), img * observed[..., None]
    box = np.ones((3, 3))
    while not known.all():
        weight = convolve(known.astype(float), box, mode="nearest")
        fresh = (weight > 0) & ~known
        for c in range(img.shape[2]):
            total = convolve(estimate[..., c] * known, box, mode="nearest")
            estimate[..., c][fresh] = total[fresh] / weight[fresh]
        known |= fresh
    return estimate


start_time = time.time()
predict, fills = [], {}
for rank in (10, 20, 40, 80):
    cp = parafac(X_img, rank, init="random", random_state=0, n_iter_max=80, tol=1e-9,
                 linesearch=True, mask=train.astype(float))
    est = tl.cp_to_tensor(cp)
    predict.append(("CP", rank, cp_numbers(X_img.shape, rank), psnr(X_img, est, val), psnr(X_img, est, test)))
    fills[("CP", rank)] = est
for r in (16, 24, 32):
    for r_p in (4, 8, 16):
        ranks = (r, r, r_p, 3)
        est = fit_tucker(X_img, ranks, mask=train, sweeps=30)
        predict.append(("Tucker", ranks, tucker_numbers(X_img.shape, ranks),
                        psnr(X_img, est, val), psnr(X_img, est, test)))
        fills[("Tucker", ranks)] = est
predict = pd.DataFrame(predict, columns=["model", "ranks", "numbers", "val_psnr", "test_psnr"])
baseline = fold(neighbour_mean(photo, labels == 0))

chosen = {m: g.loc[g.val_psnr.idxmax()] for m, g in predict.groupby("model")}
print(f"{'':22}{'numbers':>9}{'validation':>12}{'test':>8}")
for m, row in chosen.items():
    print(f"{m + ' ' + str(row.ranks):<22}{row.numbers:>9,}{row.val_psnr:>11.2f} dB{row.test_psnr:>6.2f} dB")
print(f"{'3 × 3 neighbour mean':<22}{'—':>9}{psnr(X_img, baseline, val):>11.2f} dB"
      f"{psnr(X_img, baseline, test):>6.2f} dB")
print("Better at predicting hidden pixels / Mejor prediciendo píxeles ocultos:",
      max(chosen, key=lambda m: chosen[m].test_psnr), f"({time.time() - start_time:.0f} s)")

fig, ax = plt.subplots(figsize=(8, 3.8))
for m, colour in (("CP", ACCENT), ("Tucker", CORAL)):
    g = predict[predict.model == m].sort_values("numbers")
    ax.plot(g.numbers, g.val_psnr, "o-" if m == "CP" else "s", color=colour, label=m)
    ax.scatter(chosen[m].numbers, chosen[m].val_psnr, s=160, facecolors="none", edgecolors=colour, lw=1.6)
ax.axhline(psnr(X_img, baseline, val), color=MUTED, ls="--", lw=1, label="3 × 3 neighbour mean")
ax.set_xscale("log")
ax.set_xlabel("stored numbers")
ax.set_ylabel("validation PSNR (dB)")
ax.set_title("Prediction: PSNR on pixels the model never saw")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

cp_pick, tk_pick = chosen["CP"], chosen["Tucker"]
panels = [
    ("original", photo),
    ("CP, compressed", unfold_image(rebuilt[("CP", picks["CP"].ranks)])),
    ("Tucker, compressed", unfold_image(rebuilt[("Tucker", picks["Tucker"].ranks)])),
    ("kept pixels", photo * (labels == 0)[..., None]),
    ("CP fill", unfold_image(np.where(train, X_img, fills[("CP", cp_pick.ranks)]))),
    ("Tucker fill", unfold_image(np.where(train, X_img, fills[("Tucker", tk_pick.ranks)]))),
    ("3 × 3 mean fill", unfold_image(baseline)),
]
fig, axes = plt.subplots(1, len(panels), figsize=(14, 2.6))
for ax, (title, image) in zip(axes, panels):
    ax.imshow(np.clip(image[64:192, 64:192], 0, 1))
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**The ranking flips.** Tucker, the model that needed about twice CP's storage to compress this photograph, predicts its hidden pixels clearly better. Same tensor, same fold, same two models — the job decided.

**Why.** Look at what each model says about one block $(I, J)$. Tucker with the layout modes kept whole ($R = 32$) gives **every block its own coefficients** on a small set of shared within-block patterns and colours: a hidden pixel is predicted from the observed pixels of *its own block*. CP writes a block's weight on component $r$ as a **product** $a_{Ir}\, b_{Jr}$ of a row factor and a column factor. That tie between blocks — block $(I, J)$ can only use a combination that row $I$ and column $J$ agree on — is exactly what lets CP compress, and exactly what stops it predicting. Note too that Tucker's pick is not its biggest model: more within-block patterns start fitting the noise in the training pixels, and validation catches it.

**And neither beats the neighbour mean.** Random single-pixel holes almost always have observed neighbours, and a photograph is smooth at that scale. Holes a 3 × 3 average cannot bridge — missing blocks, missing rows — are where completion by factorisation should pay; they are not tested here. That is a limit of the experiment worth saying out loud.

The blog post goes one step further and moves the tuned models to a *second* photograph, reusing the within-block and colour factors. For compression, Tucker's shared factors transfer with little loss and CP's do not; the quality a given rank buys, and so the rank itself, has to be found again on each image.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>El orden se invierte.</b> Tucker, que necesitaba el doble de almacenamiento para comprimir, predice claramente mejor los píxeles ocultos. Mismo tensor, mismo plegado, mismos modelos: decidió el trabajo.
<br><br>
<b>Por qué.</b> Tucker con los modos de disposición enteros da a <b>cada bloque sus propios coeficientes</b> sobre unos pocos patrones compartidos: un píxel oculto se predice con los píxeles observados de <i>su propio bloque</i>. CP escribe el peso de un bloque en la componente r como un <b>producto</b> a_Ir · b_Jr de un factor de fila y uno de columna. Ese vínculo entre bloques es justo lo que permite comprimir a CP y justo lo que le impide predecir. La elección de Tucker tampoco es su modelo más grande: más patrones empiezan a ajustar ruido, y la validación lo detecta.
<br><br>
<b>Ninguno supera a la media de vecinos</b>: los huecos de un píxel casi siempre tienen vecinos observados y una foto es suave a esa escala. Los huecos grandes —bloques o filas enteras— son donde la factorización debería compensar, y aquí no se prueban.</div>

## The decision, in one table / La decisión, en una tabla

<div style="height:3px;border-radius:2px;margin:1.6em 0 1.9em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

| | CP | Tucker (HOSVD to start it and to read ranks) |
|---|---|---|
| **Core** | Superdiagonal: component $r$ meets only component $r$ | Dense: any component meets any other |
| **Serves** | Interpretable parts; the lowest error per stored number — *if* degeneracy is controlled | Asymmetric relations through one shared core; predicting hidden entries; a well-conditioned compressed layer |
| **Uniqueness** | Essentially unique under Kruskal's condition | A subspace per mode; factors fixed only up to an invertible change of basis |
| **Storage** | $R \sum_n I_n$, linear in $R$ | $\prod_n R_n + \sum_n I_n R_n$; the core grows as a product |
| **Knowledge graph (Part 2)** | Untied CP-N3 ties TuckER on MRR, at ~11× the parameters | TuckER: a core paid once, not per entity |
| **Conv kernel (Part 3)** | About half Tucker-2's numbers at the same error | Orthonormal factors, no cancelling terms |
| **Photograph (Part 5)** | Compresses in about half the numbers | Predicts hidden pixels clearly better |
| **Choosing the rank** | No spectrum; over-parameterise and prune, or validate on held-out entries | HOSVD scree when there is a gap; otherwise an energy budget, a noise model, or held-out entries |
| **Typical failure** | Rank-one terms many times the tensor's norm, cancelling | A core that eats the budget before the factors can |

**The sentence to remember:** *CP compresses, Tucker predicts — and a symmetric core rarely bites, but when entities play both roles, it always does.*

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Núcleo</b>: CP superdiagonal, Tucker denso. <b>Sirve para</b>: CP, partes interpretables y el menor error por número guardado <i>si</i> se controla la degeneración; Tucker, relaciones asimétricas con un núcleo compartido, predecir entradas ocultas y una capa comprimida bien condicionada. <b>Unicidad</b>: CP esencialmente única (Kruskal); Tucker, un subespacio por modo. <b>Almacenamiento</b>: CP lineal en R; el núcleo de Tucker crece como producto. <b>Rango</b>: CP no tiene espectro (sobreparametriza y poda, o valida); Tucker usa el codo de HOSVD cuando hay hueco. <b>Fallo típico</b>: CP, términos enormes que se cancelan; Tucker, un núcleo que se come el presupuesto.
<br><br>
<b>La frase que recordar:</b> <i>CP comprime, Tucker predice; y un núcleo simétrico rara vez muerde, pero cuando las entidades tienen doble papel, muerde siempre.</i></div>

## What this notebook does not show / Lo que este cuaderno no demuestra

A careful reader should be able to list the ways each conclusion could be wrong. Here are the ones the evidence above does not rule out.

- **Weight error is not accuracy.** Lebedev et al. and Kim et al. both fine-tune after decomposing. Kim et al. report VGG-16 at 4.93× fewer FLOPs for a 0.5-point top-5 loss once fine-tuned. The weight error in Part 3 ignores the input activations and the batch norm after the layer.
- **Multiply-adds track parameters only at stride 1.** Wall-clock time can reverse the ranking: the 3×1 and 1×3 depthwise steps do little arithmetic per byte they move.
- **The knowledge-graph verdict rests on published sizes.** Each paper tuned its own embedding size, and none of the quoted tables reports CP-N3 at TuckER's parameter count. The toy world shows a mechanism, not a benchmark.
- **Tucker's core is exponential in the order.** At eight modes with every $R_n = 10$ the core alone is $10^8$ numbers. Everything here had four modes or fewer.
- **The image test hides single pixels,** and uses one pixel split and one random start per CP rank; the blog post uses three of each and the one-standard-error rule to choose ranks. The 8 × 8 block size is fixed, not tuned.
- **Your live fits are short.** 80–100 ALS iterations, where the cached runs used 1,000. CP's error was still falling; the conclusions only get stronger with more iterations, but the numbers you printed are not converged.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:15px 19px;margin:1.8em 0 2.2em;font:400 14.5px/1.8 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Un lector cuidadoso debe poder enumerar cómo podría fallar cada conclusión. <b>El error en los pesos no es exactitud</b>: los trabajos de compresión reajustan después. <b>Multiplicaciones y parámetros solo coinciden con paso 1</b>, y el tiempo real puede invertir el orden. <b>El veredicto en grafos se basa en tamaños publicados</b>, y el mundo de juguete muestra un mecanismo, no un benchmark. <b>El núcleo de Tucker es exponencial en el orden</b>. <b>La prueba de imagen oculta píxeles sueltos</b> con una sola partición y un solo arranque. <b>Tus ajustes en vivo son cortos</b> (80–100 iteraciones frente a 1.000): las conclusiones solo se refuerzan con más iteraciones, pero los números impresos no están convergidos.</div>

## Questions to test yourself / Preguntas para ponerte a prueba

Answer each in two or three sentences before opening it. Each one is the kind of question an examiner — or a colleague reviewing your design doc — would ask.

<details>
<summary><b>1.</b> A colleague proposes DistMult for a citation graph (<i>paper cites paper</i>). What do you measure on the data before arguing? / Un colega propone DistMult para un grafo de citas. ¿Qué mides antes de discutir?</summary>

The two-role fraction of `cites`: almost every paper both cites and is cited, so nearly every pair is exposed, and DistMult will score *(A cites B)* exactly as high as *(B cites A)* — ranking the future paper among the references of the past one. Use a model with an asymmetric core (ComplEx, CP with reciprocal relations, TuckER).

ES: la fracción de doble papel de `cites`. Casi todo artículo cita y es citado, así que casi todos los pares están expuestos y DistMult puntuará igual *(A cita B)* que *(B cita A)*. Usa un modelo con núcleo asimétrico.

</details>

<details>
<summary><b>2.</b> Why does a CP term many times larger than ‖K‖ make fine-tuning unstable, when the reconstruction itself is accurate? / ¿Por qué un término CP muchas veces mayor que ‖K‖ vuelve inestable el reajuste?</summary>

Because the accuracy comes from cancellation between large terms. A gradient step changes the factors of each term, and a relative change $\epsilon$ in one factor of a term of norm $c\|\mathcal{K}\|$ moves the kernel by about $\epsilon c \|\mathcal{K}\|$: the cancellation is broken $c$ times faster than the step size suggests. Orthonormal Tucker factors bound the core's norm by $\|\mathcal{K}\|$, so they have no such amplification.

ES: porque la exactitud viene de cancelar términos grandes. Un cambio relativo ε en un factor de un término de norma c‖K‖ mueve el kernel ≈ εc‖K‖: la cancelación se rompe c veces más rápido de lo que sugiere el paso. Los factores ortonormales de Tucker acotan la norma del núcleo.

</details>

<details>
<summary><b>3.</b> The HOSVD bound held at the 90% ranks. Why is it still a poor way to choose ranks for this kernel? / La cota de HOSVD se cumplió. ¿Por qué sigue siendo una mala forma de elegir rangos aquí?</summary>

The bound controls *weight* error given the ranks, but the spectrum has no gap, so any threshold is arbitrary and the one that looks safe (90%) barely compresses. What matters is accuracy after fine-tuning, which the spectrum does not see; a noise model (VBMF) or a held-out accuracy measurement answers the real question.

ES: la cota controla el error en los pesos dados los rangos, pero el espectro no tiene hueco, así que cualquier umbral es arbitrario y el que parece seguro apenas comprime. Lo que importa es la exactitud tras reajustar.

</details>

<details>
<summary><b>4.</b> For a matrix $UV^\top$, weight decay on the factors is the nuclear norm. For a 3-way CP it becomes $3\lambda_r^{2/3}$. What changes, and why is the second more aggressive about small terms? / Para una matriz, weight decay es la norma nuclear; para CP de 3 modos se vuelve 3λ_r^(2/3). ¿Qué cambia?</summary>

By AM–GM, weight decay on a term's $N$ factors costs $N\lambda_r^{2/N}$ at a balanced point. For $N = 2$ that is $2\lambda_r$, the nuclear norm: convex, with the *same* slope for every singular value, so its proximal map soft-thresholds them all by one fixed amount — small ones go to zero, and every survivor is shrunk by just as much. For $N = 3$ the cost $3\lambda_r^{2/3}$ is concave, and its slope $2\lambda_r^{-1/3}$ grows without bound as $\lambda_r \to 0$ and fades for large terms: small terms are pushed out hard while large ones are barely biased. The order of the tensor changes the shape of the penalty.

ES: por AM–GM, weight decay cuesta Nλ_r^(2/N). Con N = 2 es la norma nuclear, convexa y con la <i>misma</i> pendiente para todos los valores singulares: los umbraliza todos por la misma cantidad. Con N = 3, 3λ_r^(2/3) es cóncava, con pendiente 2λ_r^(−1/3) que crece sin límite cerca de cero y se apaga para términos grandes: expulsa con fuerza los pequeños y apenas sesga los grandes.

</details>

<details>
<summary><b>5.</b> In Part 5, would making the blocks 16 × 16 help CP predict, or Tucker compress? Predict, then try it. / En la Parte 5, ¿bloques de 16 × 16 ayudarían a CP a predecir o a Tucker a comprimir? Predice y pruébalo.</summary>

Larger blocks shrink the layout modes to 16 × 16 and grow the within-block mode to 256. Tucker's core over the layout modes gets four times smaller, so Tucker compresses more cheaply; each block now carries more pixels, so a block's own coefficients are estimated from more observations. CP still ties blocks through $a_{Ir} b_{Jr}$. Try it: `fold(photo, block=16)` and rerun the two sweeps with the ranks scaled to the new shape.

ES: bloques mayores reducen los modos de disposición a 16 × 16 y amplían el modo interno a 256. El núcleo de Tucker sobre la disposición se hace cuatro veces menor, así que comprime más barato. CP sigue atando bloques con a_Ir·b_Jr. Pruébalo con `fold(photo, block=16)`.

</details>

## Sources and next steps / Fuentes y siguientes pasos

**The post this notebook works through.** [CP or Tucker in practice](https://project-delphi.github.io/ml-blog/posts/cp-or-tucker-in-practice/) — the full-scale experiments, three seeds each, the transfer test on a second photograph, and the complete pruning grid. Its prequel [CP or Tucker](https://project-delphi.github.io/ml-blog/posts/cp-or-tucker/) derives uniqueness, rotational freedom and storage; [What a tensor factorization buys you](https://project-delphi.github.io/ml-blog/posts/uses-of-tensor-factorizations/) walks the CP convolution chain on a synthetic kernel.

**Papers, in the order this notebook used them.**

- Bordes et al. (2013), [Translating embeddings for modeling multi-relational data](https://proceedings.neurips.cc/paper/2013/hash/1cecc7a77928ca8133fa24680a88d2f9-Abstract.html) — FB15k. Toutanova & Chen (2015), [Observed versus latent features for knowledge base and text inference](https://aclanthology.org/W15-4007/) — FB15k-237.
- Yang et al. (2015), [DistMult](https://arxiv.org/abs/1412.6575); Trouillon et al. (2016), [ComplEx](https://arxiv.org/abs/1606.06357); Nickel et al. (2011), [RESCAL](https://icml.cc/2011/papers/438_icmlpaper.pdf); Balažević et al. (2019), [TuckER](https://arxiv.org/abs/1901.09590); Lacroix et al. (2018), [Canonical tensor decomposition for knowledge base completion](https://arxiv.org/abs/1806.07297) — CP-N3, reciprocal relations. Ruffinelli et al. (2020), [You CAN teach an old dog new tricks!](https://openreview.net/forum?id=BkxSmlBFvr) — the re-tuned baselines, with MRRs from the [LibKGE results table](https://github.com/uma-pi1/kge).
- He et al. (2016), [ResNet](https://arxiv.org/abs/1512.03385); Lebedev et al. (2015), [Speeding-up CNNs using fine-tuned CP-decomposition](https://arxiv.org/abs/1412.6553); Kim et al. (2016), [Compression of deep CNNs for fast and low power mobile applications](https://arxiv.org/abs/1511.06530) — Tucker-2; Phan et al. (2020), [Stable low-rank tensor decomposition for compression of CNNs](https://arxiv.org/abs/2008.05441) — CP degeneracy.
- De Lathauwer, De Moor & Vandewalle (2000), [A multilinear singular value decomposition](https://doi.org/10.1137/S0895479896305696) — HOSVD and its truncation bound. Sidiropoulos & Bro (2000), [On the uniqueness of multilinear decomposition of N-way arrays](https://doi.org/10.1002/1099-128X%28200005/06%2914:3%3C229::AID-CEM587%3E3.0.CO;2-N) — Kruskal's condition for N modes.
- Kossaifi et al. (2019), [TensorLy: Tensor learning in Python](https://jmlr.org/papers/v20/18-277.html).

**Continue with** [deep dive 14, CP factorization](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/14-cp-factorization.ipynb), to fit CP by hand and read its components, or [deep dive 15, generalized CP](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/15-generalized-cp.ipynb), for the question every fit here took for granted: what squared error assumes.

🇪🇸 Las fuentes siguen el orden del cuaderno: la entrada del blog (en inglés) con los experimentos completos, los artículos de grafos de conocimiento, los de compresión de convoluciones, HOSVD y la condición de Kruskal, y TensorLy. Para seguir: el estudio 14 (CP a mano) o el 15 (CP generalizado).

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#0891b2,rgba(8,145,178,0))"></div>

## Done with this deep dive / Fin de este estudio a fondo

Next deep dive / Siguiente estudio a fondo: **20 · AlphaTensor: multiplying matrices as a game / AlphaTensor: multiplicar matrices como un juego** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/20-alphatensor.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)